# MangoLeaf AI V4.3.0: one seven-class combined experiment

**Experimental candidate template; not trained, evaluated, adopted, or deployed.** This is one model for all seven V4.1.1 classes. It is not a leaf/fruit specialist workflow. The only training arm is `combined`: the unchanged flip/rotation/zoom/shear/brightness recipe plus reviewed scene replacement at 30% of approved white-background presentations and modest local lighting at 30% of all training presentations. These two probabilities act independently. A larger scene pool does not increase the number of approved foreground photos or mean that 30% of all training photos receive a scene.

The saved V4.1.1 `dataset_split.zip` is copied byte-for-byte. Its exact historical 70/15/15 membership remains 1,819 training, 391 validation, and 393 test images, with all seven classes in each subset. Image decoding, 224 x 224 preprocessing, batch size 32, ImageNet MobileNetV3Large, balanced class weights, 5 head epochs at 0.001, 10 fine-tune epochs at 0.00001, per-stage minimum validation-loss checkpoints, and validation-only temperature calibration retain the V4.1.1 recipe. Batch-normalization statistics stay frozen in fine-tuning. Temperature is folded into the final logits once.

Run **Steps 1–24** in order in a Colab GPU runtime. Review every scene page, every training white-background page, every mask composite, and all lighting previews before approving their switches. Then draw the training diagnostics in **Steps 25–26**. Run the separate `MangoLeaf_AI_Threshold_Fitting_v4_3_0.ipynb`, using the printed completed training path. Return here for **Steps 27–30** to draw final figures from its frozen reports. This notebook never predicts on test or OOD images, refits rejection gates, or retrains for figures. Grad-CAM uses seven validation examples, one per class, and is a diagnostic rather than accuracy evidence.

There are exactly two V4.3.0 notebooks: this main training notebook, including figures, and the threshold-fitting notebook. Keep full Drive run folders and their hashes. V4.3.0 is a workflow label; no shipped app/model version changes. The historical test/OOD collections influenced earlier development, so future accuracy and field/OOD evaluation are still required before adoption.

## Drive inputs and durable outputs

Set `V4_1_1_TRAINING_RUN` to the exact completed V4.1.1 training folder; it must contain `training_complete.json`, `plan.json`, `input_manifest.json`, `dataset_split.zip`, and `duplicate_audit.json`. The original raw dataset and v3 split folders are not required. Optional `white_background_review.json`, `mask_review.json`, and `masks/*.png` support reuse after verification; old review decisions are starting suggestions and must be checked again. Never point at the parent `training/` folder or automatically choose a latest run.

The actual V4.1.1 notebook scene root is `MyDrive/mangoleaf_background_photos/`, with a reported 62 original photos. Add `MyDrive/mangoleaf_new_background_photos/`, reported as 138 fresh photos, for 200 reported input files. `mango_backgrounds` is not the recorded V4.1.1 folder name. The audit records supported files, readable images, byte hashes, decoded RGB identities, repeated content, similarity flags, exclusions, and final approved unique counts. Reported counts are expectations to check, not assumed evidence. An unexpected count requires a written explanation and explicit acknowledgement in Step 7.

Outputs live only in `MyDrive/mangoleaf_experiments/v4.3.0/training/v4-3-0-training-<UTC>-<unique>/`. The run contains the reused ZIP, source provenance, frozen approved `backgrounds/`, input/scene inventories, human review files, masks, a frozen 15-epoch scene schedule, `plan.json`, `recovery_state.json`, `arms/combined/`, `figures/`, and finally `training_complete.json`. No app files or historical runs are overwritten. Available old/new OOD folders are read only for input-overlap audit; their predictions remain in the separate threshold notebook.

### Step 1: Mount Drive and freeze settings

- **Purpose:** Load the seven-class order, source paths, runtime, and unchanged V4.1.1 recipe.
- **Reads:** Exact saved V4.1.1 run and both scene roots; truthful source and rights notes.
- **Saves:** Printed settings only.
- **Check:** Fill both required path/notes fields. Verify source completion hash against its recorded provenance, GPU, classes, and 62 + 138 expectations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import csv, hashlib, json, math, os, random, shutil, subprocess, sys, uuid, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps, ImageDraw

SEED = 42
CLASS_NAMES = ['anthracnose', 'bacterial_black_spot', 'bacterial_canker',
               'healthy_fruit', 'healthy_leaf', 'mango_scab', 'sooty_mold']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CONFIDENCE_THRESHOLD = 0.70
MIN_ID_RETENTION = 0.95  # Accepted mango images; not necessarily correct diagnoses.
MODEL_VERSION = 'model-v4.3.0-candidate'
ARMS = ('combined',)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
PREPROCESSING = 'TensorFlow RGB decode (Pillow RGB with EXIF transpose for WebP); bilinear 224x224, no antialias; float32 0-255; model rescales internally'
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.keras.mixed_precision.set_global_policy('float32')
RUNTIME = {'python': sys.version, 'tensorflow': tf.__version__, 'keras': tf.keras.__version__,
           'numpy': np.__version__, 'pillow': Image.__version__}
print('Runtime:', RUNTIME)

# Choose the exact saved source; a run folder is required, not a parent directory.
V4_1_1_TRAINING_RUN = ''
# This is the completed V4.1.1 record published with the historical experiment.
SOURCE_COMPLETION_SHA256 = '2c62e4289c449ed5f1b18c1690ddf0341dc07a3cb36ae5807def42babbe6bd16'
# If using a different genuine completed V4.1.1 run, record its externally saved
# completion hash and explain that source choice; never edit hashes to hide changes.
SOURCE_CHOICE_NOTES = 'Reuse the completed V4.1.1 seven-class historical experiment.'
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.3.0')
OUTPUT_ROOT = EXPERIMENT_ROOT / 'training'
BACKGROUND_DIR = Path('/content/drive/MyDrive/mangoleaf_background_photos')
NEW_BACKGROUND_DIR = Path('/content/drive/MyDrive/mangoleaf_new_background_photos')
BACKGROUND_ROOTS = {'old': BACKGROUND_DIR, 'new': NEW_BACKGROUND_DIR}
EXPECTED_BACKGROUND_COUNTS = {'old': 62, 'new': 138}
OLD_OOD_DIR = Path('/content/drive/MyDrive/mango_ood_test')
NEW_OOD_DIR = Path('/content/drive/MyDrive/mango_ood_test_new_photos')
BACKGROUND_NOTES = ''  # Record sources, dates, usage rights, and the new collection.
REUSE_REVIEWED_MASKS = True  # Valid saved masks avoid repeating segmentation work.
BLEND_PROBABILITY = 0.30  # Among approved white-background images, per epoch.
LOCAL_LIGHT_PROBABILITY = 0.30  # Among all training images, per epoch.
LIGHT_AMPLITUDE = (18.0, 40.0)  # Pixel change, bounded below clipping at 0..255.
LIGHT_RADIUS = (0.18, 0.35)  # Gaussian radius relative to 224-pixel side.
if not 0 < BLEND_PROBABILITY < 1 or not 0 < LOCAL_LIGHT_PROBABILITY < 1:
    raise ValueError('Keep original presentations alongside each augmentation effect.')
PHASH_DISTANCE = 5
STAGE1_EPOCHS, STAGE1_LR = 5, 1e-3
STAGE2_EPOCHS, STAGE2_LR = 10, 1e-5
AUG_ROTATION_DEG, AUG_ZOOM, AUG_BRIGHTNESS, AUG_SHEAR = 30, 0.20, 0.20, 0.20
AUG_FLIP = 'horizontal_and_vertical'
QUANTIZE_FLOAT16 = False
EVALUATION_STATUS = 'previously_used'
EVALUATION_NOTES = 'The original v3 test/OOD report images influenced earlier development.'

# Install hashing support without changing Colab's TensorFlow or NumPy versions.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'ImageHash==4.3.2'], check=True)
import imagehash
print('Class order:', CLASS_NAMES)
print('Training output root:', OUTPUT_ROOT)
print('Arms:', ARMS)

SOURCE_RUN = Path(V4_1_1_TRAINING_RUN.strip()) if V4_1_1_TRAINING_RUN.strip() else None
if SOURCE_RUN is not None and not SOURCE_RUN.is_absolute():
    raise ValueError('V4_1_1_TRAINING_RUN requires the full absolute saved run path.')
if SOURCE_RUN is not None:
    SOURCE_RUN = SOURCE_RUN.resolve()
if not (len(SOURCE_COMPLETION_SHA256) == 64 and
                       all(c in '0123456789abcdef' for c in SOURCE_COMPLETION_SHA256)):
    raise ValueError('SOURCE_COMPLETION_SHA256 must be the recorded lowercase SHA-256.')
print('Saved V4.1.1 source:', SOURCE_RUN)
print('Scene expectations:', EXPECTED_BACKGROUND_COUNTS, 'total:', sum(EXPECTED_BACKGROUND_COUNTS.values()))

### Step 2: Define safety, hashing, and image helpers

- **Purpose:** Verify artifact bytes, safely extract only manifest images, and retain the exact model input decoding.
- **Reads:** Libraries and image-size settings from Step 1.
- **Saves:** Helper functions only.
- **Check:** SHA-256 identifies bytes; decoded RGB identity finds content repeated in another encoding; perceptual hashes are review flags.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def object_hash(value):
    text = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(text.encode()).hexdigest()


def write_json_once(path, value):
    with Path(path).open('x', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
        handle.write('\n')


def list_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f'Image folder missing: {folder}')
    paths = sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        raise ValueError(f'No supported images in {folder}')
    return paths


def verify_records_unchanged(rows):
    for row in rows:
        if sha256_file(row['path']) != row['sha256']:
            raise RuntimeError('Image changed: ' + row['path'])


def inside(root, relative):
    relative = PurePosixPath(relative)
    if relative.is_absolute() or '..' in relative.parts or '\\' in str(relative):
        raise ValueError('Unsafe relative path: ' + str(relative))
    target = (Path(root) / str(relative)).resolve()
    if not target.is_relative_to(Path(root).resolve()) or target == Path(root).resolve():
        raise ValueError('Path must stay inside its output folder.')
    return target


def extract_dataset(archive, rows, destination):
    # Extract only recorded images, then verify their contents before inference.
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    expected = {r['relativePath']: r for r in rows}
    if len(expected) != len(rows):
        raise ValueError('The image manifest repeats a destination path.')
    with zipfile.ZipFile(archive) as bundle:
        names = bundle.namelist()
        if len(names) != len(set(names)) or set(names) != set(expected):
            raise ValueError('Dataset ZIP entries differ from the image manifest.')
        for name, row in expected.items():
            target = inside(destination, name)
            if bundle.getinfo(name).file_size != row['bytes']:
                raise ValueError('Image size changed in dataset ZIP: ' + name)
            target.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(name) as source, target.open('xb') as output:
                shutil.copyfileobj(source, output)
            if sha256_file(target) != row['sha256']:
                raise ValueError('Image contents changed in dataset ZIP: ' + name)
    return [{**r, 'path': str(inside(destination, r['relativePath']))} for r in rows]


def verify_artifacts(folder, artifacts):
    for relative, expected in artifacts.items():
        path = inside(folder, relative)
        if not path.is_file() or sha256_file(path) != expected:
            raise RuntimeError('Missing or changed saved artifact: ' + relative)


def load_pixels(path):
    # Keep V4's RGB pixels; WebP uses Pillow because tf.io.decode_image lacks it.
    if Path(path).suffix.lower() == '.webp':
        # TensorFlow's generic decoder does not support WebP. Match threshold notebook.
        with Image.open(path) as picture:
            pixels = np.asarray(ImageOps.exif_transpose(picture).convert('RGB'), dtype=np.float32)
    else:
        pixels = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        pixels.set_shape((None, None, 3))
    return tf.image.resize(pixels, IMAGE_SIZE, method='bilinear',
                           preserve_aspect_ratio=False, antialias=False)


def predict_records(model, rows):
    # Training calibration may read validation only; final test/OOD inference belongs in threshold fitting.
    if any(r['subset'] != 'val' for r in rows):
        raise ValueError('Training prediction helper only accepts validation images.')
    output = []
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        verify_records_unchanged(batch)
        pixels = tf.stack([load_pixels(r['path']) for r in batch])
        values = model(pixels, training=False).numpy()
        if values.shape != (len(batch), len(CLASS_NAMES)) or not np.isfinite(values).all():
            raise ValueError('The model must return seven finite probabilities per image.')
        output.extend(values.tolist())
    if not output:
        raise ValueError('Cannot predict an empty image collection.')
    return np.asarray(output, dtype=np.float32)


def weight_digest(model):
    digest = hashlib.sha256()
    for variable in model.weights:
        values = variable.numpy()
        digest.update(str(values.shape).encode())
        digest.update(str(values.dtype).encode())
        digest.update(values.tobytes())
    return digest.hexdigest()



def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def copy_verified(source, destination, expected):
    # Copy through a unique partial file, so a interrupted copy cannot look complete.
    source, destination = Path(source), Path(destination)
    if sha256_file(source) != expected:
        raise RuntimeError('Source artifact changed: ' + str(source))
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.partial-' + uuid.uuid4().hex[:8])
    with source.open('rb') as incoming, temporary.open('xb') as outgoing:
        shutil.copyfileobj(incoming, outgoing)
    if sha256_file(temporary) != expected or sha256_file(source) != expected:
        raise RuntimeError('Artifact changed while copying: ' + str(source))
    if destination.exists():
        raise FileExistsError('Refuse to overwrite frozen evidence: ' + str(destination))
    temporary.rename(destination)


def rgb_identity(path):
    # Use the scene loader's first-frame RGB decoding without EXIF rotation.
    # Include dimensions so equal byte arrays with different shapes cannot collide.
    with Image.open(path) as picture:
        pixels = np.asarray(picture.convert('RGB'), dtype=np.uint8)
    digest = hashlib.sha256()
    digest.update(json.dumps(list(pixels.shape), separators=(',', ':')).encode())
    digest.update(pixels.tobytes())
    return digest.hexdigest(), list(pixels.shape[:2][::-1])


def canonical_phash(path):
    # Similarity flags catch likely repeated views; a person decides if they are distinct.
    with Image.open(path) as picture:
        picture = picture.convert('L')
        hashes = []
        for k in range(4):
            rotated = picture.rotate(90 * k, expand=True)
            hashes.extend([imagehash.phash(rotated), imagehash.phash(ImageOps.mirror(rotated))])
    return min(hashes, key=str)

### Step 3: Define saved-source and review checks

- **Purpose:** Verify the completed V4.1.1 source, exact 70/15/15 identities, and optional saved review/mask provenance.
- **Reads:** Source completion record, manifest, plan, ZIP, duplicate audit, and optional reviews/masks.
- **Saves:** Helpers only; source files stay read-only.
- **Check:** Absent optional records require fresh review/generation. A present but changed requested reuse record/mask blocks reuse: investigate or explicitly choose REUSE_REVIEWED_MASKS=False. Invalid source split, recipe, or completion hashes block the experiment.

In [ ]:
# Freeze the historical identity, not a newly randomized split.
EXPECTED_BASELINE_IDENTITY = '3ca5a4625524eb5546d999f8a1ebcd15aac1cb7231b48963fdc565aee77559b8'
EXPECTED_COUNTS = {'train': 1819, 'val': 391, 'test': 393}


def load_source_contract(folder):
    complete_path = folder / 'training_complete.json'
    if sha256_file(complete_path) != SOURCE_COMPLETION_SHA256:
        raise ValueError('V4.1.1 completion record differs from recorded provenance; inspect the source before proceeding.')
    complete = read_json(complete_path)
    if (complete['modelVersion'] != 'model-v4.1.1-candidate' or
        complete['classes'] != CLASS_NAMES or complete.get('testPredictionsRun') is not False):
        raise ValueError('Select the completed seven-class V4.1.1 training source.')
    for name, key in (('plan.json', 'planSha256'), ('input_manifest.json', 'inputManifestSha256'),
                      ('dataset_split.zip', 'datasetSplitSha256')):
        declared = complete['artifacts'].get(name)
        if declared != complete[key] or sha256_file(folder/name) != declared:
            raise ValueError('V4.1.1 source artifact disagrees with completion: ' + name)
    if sha256_file(folder/'duplicate_audit.json') != complete['artifacts']['duplicate_audit.json']:
        raise ValueError('V4.1.1 duplicate audit changed.')
    old_plan, old_manifest = read_json(folder/'plan.json'), read_json(folder/'input_manifest.json')
    recipe = {'classes': CLASS_NAMES, 'seed': SEED, 'batchSize': BATCH_SIZE,
        'imageSize': list(IMAGE_SIZE), 'stage1': [STAGE1_EPOCHS, STAGE1_LR],
        'stage2': [STAGE2_EPOCHS, STAGE2_LR], 'preprocessing': PREPROCESSING,
        'augmentation': [AUG_FLIP, AUG_ROTATION_DEG, AUG_ZOOM, AUG_SHEAR, AUG_BRIGHTNESS],
        'splitFractions': [0.70, 0.15, 0.15], 'baselineIdentity': EXPECTED_BASELINE_IDENTITY,
        'blendProbability': BLEND_PROBABILITY, 'localLightProbability': LOCAL_LIGHT_PROBABILITY,
        'lightAmplitude': list(LIGHT_AMPLITUDE), 'lightRadius': list(LIGHT_RADIUS),
        'confidenceThreshold': CONFIDENCE_THRESHOLD, 'minimumValidationRetention': MIN_ID_RETENTION}
    if any(old_plan.get(key) != value for key, value in recipe.items()):
        raise ValueError('Source training recipe differs from V4.1.1; do not silently change the comparison.')
    rows = old_manifest['images']
    if old_manifest['classes'] != CLASS_NAMES or len(rows) != sum(EXPECTED_COUNTS.values()):
        raise ValueError('Source manifest is not the complete seven-class historical split.')
    if (len({r['relativePath'] for r in rows}) != len(rows) or
        len({r['sha256'] for r in rows}) != len(rows)):
        raise ValueError('Source manifest repeats an image path or file content.')
    for row in rows:
        if row['subset'] not in EXPECTED_COUNTS or row['label'] not in CLASS_NAMES:
            raise ValueError('Unknown source subset or label.')
        if PurePosixPath(row['relativePath']).parts[:2] != (row['subset'], row['label']):
            raise ValueError('Source image path disagrees with its subset/label.')
        if row['bytes'] <= 0:
            raise ValueError('Source image size must be positive.')
    counts = {subset: dict(Counter(r['label'] for r in rows if r['subset'] == subset))
              for subset in EXPECTED_COUNTS}
    if ({subset: sum(value.values()) for subset, value in counts.items()} != EXPECTED_COUNTS or
        counts != old_manifest['counts'] or any(set(value) != set(CLASS_NAMES) for value in counts.values())):
        raise ValueError('The historical split counts or seven-class coverage changed.')
    identity = object_hash(sorted((r['subset'], r['label'], r['sha256']) for r in rows))
    if identity != EXPECTED_BASELINE_IDENTITY:
        raise ValueError('Saved split membership differs from the recorded historical content identity.')
    return complete, old_plan, old_manifest, counts


def load_reusable_reviews(folder, complete, old_manifest):
    # Optional reviews are accepted only by their recorded bytes and matching source images.
    # Invalid or absent records do not approve an image or mask in the new experiment.
    audit = {'sourceRun': str(folder), 'records': {}, 'maskCandidates': [], 'limits':
             'Imported white decisions are suggestions; all current white and mask pages require review.'}
    white, mask = None, None
    for name, key in (('white_background_review.json', 'whiteReviewSha256'),
                      ('mask_review.json', 'maskReviewSha256')):
        path = folder/name
        expected = complete.get(key)
        valid = (path.is_file() and expected and complete['artifacts'].get(name) == expected and
                 sha256_file(path) == expected)
        audit['records'][name] = {'validHash': bool(valid), 'expectedSha256': expected}
        if path.is_file() and not valid:
            if REUSE_REVIEWED_MASKS:
                raise ValueError('Requested source review reuse has changed hashes: ' + name +
                                 '. Investigate it, or explicitly set REUSE_REVIEWED_MASKS=False for fresh mask generation.')
            print('Ignoring invalid optional source review for fresh generation:', name)
        elif not path.is_file():
            print('Optional saved review absent; fresh review/generation required:', name)
        if valid:
            if name.startswith('white_'):
                white = read_json(path)
            else:
                mask = read_json(path)
    if white is not None and (white.get('allTrainImagesReviewed') is not True or
        white.get('inputManifestSha256') != complete['inputManifestSha256']):
        audit['records']['white_background_review.json']['reason'] = 'Review does not identify the saved training manifest.'
        if REUSE_REVIEWED_MASKS:
            raise ValueError('Requested source white review does not identify the saved manifest. Investigate or choose fresh mask generation explicitly.')
        white = None
    if mask is not None and (white is None or mask.get('whiteReviewSha256') != complete['whiteReviewSha256']):
        audit['records']['mask_review.json']['reason'] = 'Mask review does not link the valid source white review.'
        if REUSE_REVIEWED_MASKS:
            raise ValueError('Requested source mask review does not link a valid source white review. Investigate or choose fresh mask generation explicitly.')
        mask = None
    source_rows = {r['relativePath']: r for r in old_manifest['images'] if r['subset'] == 'train'}
    if white is not None:
        paths = white.get('eligibleRelativePaths', [])
        if len(paths) != len(set(paths)) or set(paths) - set(source_rows):
            raise ValueError('Saved white review refers to unknown or repeated training images.')
    reusable = {}
    if mask is not None and REUSE_REVIEWED_MASKS:
        for row in mask['masks']:
            current = source_rows.get(row.get('relativePath'))
            result = {'id': row.get('id'), 'relativePath': row.get('relativePath'), 'reused': False}
            # Locate only inside the selected run; never follow a saved absolute maskPath.
            key = str(row.get('id', ''))
            if len(key) != 20 or any(c not in '0123456789abcdef' for c in key):
                result['reason'] = 'Invalid source mask ID.'
            elif current is None or any(row.get(k) != current[k] for k in ('label', 'sha256', 'bytes')):
                result['reason'] = 'Source mask image identity differs from historical training image.'
            elif row.get('useForBlend') is not True:
                result['reason'] = 'Source mask was rejected; generate and review a fresh candidate if now eligible.'
            else:
                path = inside(folder, 'masks/' + key + '.png')
                try:
                    if not path.is_file() or sha256_file(path) != row['maskSha256']:
                        raise ValueError('Saved mask bytes are absent or changed.')
                    with Image.open(path) as picture:
                        if picture.format != 'PNG' or picture.mode != 'L':
                            raise ValueError('Saved mask must be a grayscale PNG.')
                        picture.load()
                        mask_size = picture.size
                    # The source image is checked after ZIP extraction in Step 5.
                    reusable[current['sha256']] = {**row, 'sourceMaskPath': str(path), 'maskSize': mask_size}
                    result.update({'reused': True, 'maskSha256': row['maskSha256']})
                except (OSError, ValueError, KeyError) as error:
                    result['reason'] = str(error)
            audit['maskCandidates'].append(result)
            if (not result['reused'] and row.get('useForBlend') is True):
                raise ValueError('Requested source mask reuse failed for ' + str(row.get('relativePath')) + ': ' +
                                 result['reason'] + ' Investigate, or explicitly set REUSE_REVIEWED_MASKS=False for fresh masks.')
    return white, reusable, audit

### Step 4: Define scene content and overlap audits

- **Purpose:** Audit both required scene pools and flag unsuitable repeated content before training.
- **Reads:** Old/new scene folders, historical mango images, and any available OOD audit folders.
- **Saves:** Audit helpers only.
- **Check:** All scenes are decoded. Duplicate content and mango/OOD overlap are excluded explicitly; perceptual similarity still needs human review.

In [ ]:
def require_output_disjoint(output, inputs):
    destination = Path(output).resolve()
    for folder in inputs:
        source = Path(folder).resolve()
        if destination == source or destination in source.parents or source in destination.parents:
            raise ValueError(f'Output and input trees overlap: {destination} / {source}')


def inventory_scenes(roots):
    images, summaries = [], {}
    for pool, root in roots.items():
        paths = list_images(root)  # Both required roots must exist and contain image files.
        for path in paths:
            relative = path.relative_to(root).as_posix()
            scoped = pool + '/' + relative
            key = hashlib.sha256(scoped.encode()).hexdigest()[:20]
            row = {'id': key, 'pool': pool, 'sourceRoot': str(root), 'sourcePath': str(path),
                   'sourceRelativePath': relative, 'relativePath': scoped, 'path': str(path),
                   'sha256': sha256_file(path), 'bytes': path.stat().st_size, 'readable': False}
            try:
                row['decodedRgbSha256'], row['dimensions'] = rgb_identity(path)
                row['phash'] = str(canonical_phash(path))
                row['readable'] = True
            except (OSError, ValueError) as error:
                row['decodeError'] = str(error)
            images.append(row)
        selected = [r for r in images if r['pool'] == pool]
        summaries[pool] = {'expectedFiles': EXPECTED_BACKGROUND_COUNTS[pool],
                           'supportedFiles': len(selected), 'readableImages': sum(r['readable'] for r in selected)}
    if len({r['id'] for r in images}) != len(images):
        raise ValueError('Scene review IDs collide.')
    seen_bytes, seen_pixels = {}, {}
    similarities = []
    for row in images:
        if not row['readable']:
            row['automaticExclusion'] = 'unreadable image'
            continue
        previous = seen_bytes.get(row['sha256']) or seen_pixels.get(row['decodedRgbSha256'])
        if previous:
            row['automaticExclusion'] = 'repeated byte or decoded RGB content'
            row['duplicateOf'] = previous['id']
        else:
            row['automaticExclusion'] = None
            seen_bytes[row['sha256']] = row
            seen_pixels[row['decodedRgbSha256']] = row
        for other in images:
            if other['id'] == row['id']:
                break
            if not other['readable'] or other.get('automaticExclusion'):
                continue
            distance = imagehash.hex_to_hash(row['phash']) - imagehash.hex_to_hash(other['phash'])
            if distance <= PHASH_DISTANCE and row.get('duplicateOf') != other['id']:
                similarities.append({'first': other['id'], 'second': row['id'], 'distance': int(distance)})
    return {'roots': {k: str(v) for k, v in roots.items()}, 'expectedTotalFiles': sum(EXPECTED_BACKGROUND_COUNTS.values()),
            'supportedFiles': len(images), 'readableImages': sum(r['readable'] for r in images),
            'uniqueByteHashes': len({r['sha256'] for r in images if r['readable']}),
            'uniqueDecodedImages': len({r['decodedRgbSha256'] for r in images if r['readable']}),
            'automaticExclusions': sum(bool(r['automaticExclusion']) for r in images),
            'poolCounts': summaries, 'similarityFlags': similarities, 'images': images}


def inventory_optional_ood(folder, kind):
    if not folder.is_dir():
        print('Optional overlap-audit folder absent:', folder)
        return []
    rows = []
    for path in list_images(folder):
        decoded, dimensions = rgb_identity(path)
        rows.append({'path': str(path), 'relativePath': path.relative_to(folder).as_posix(),
                     'kind': kind, 'sha256': sha256_file(path), 'decodedRgbSha256': decoded,
                     'dimensions': dimensions, 'bytes': path.stat().st_size})
    return rows


def audit_scene_overlap(dataset, scenes, ood):
    # Decode complete contents before approving scenes; no model predictions are made.
    protected_bytes, protected_pixels, protected_hashes = {}, {}, []
    for row in dataset + ood:
        kind = 'mango-' + row['subset'] if 'subset' in row else row['kind']
        location = {'kind': kind, 'relativePath': row['relativePath']}
        decoded = row.get('decodedRgbSha256') or rgb_identity(row['path'])[0]
        protected_bytes.setdefault(row['sha256'], []).append(location)
        protected_pixels.setdefault(decoded, []).append(location)
        protected_hashes.append((canonical_phash(row['path']), location))
    conflicts, similarity_flags = [], []
    for row in scenes:
        if row['readable']:
            matched = protected_bytes.get(row['sha256'], []) or protected_pixels.get(row['decodedRgbSha256'], [])
            if matched:
                row['automaticExclusion'] = 'overlap with mango split or OOD audit image'
                row['overlapLocations'] = matched
                conflicts.append({'sceneId': row['id'], 'matches': matched})
            # Perceptual similarity does not establish leakage: record it for review.
            # Compare scenes with every mango split and available OOD photo, including
            # validation/test identities, before approving the scene pool.
            hashed = imagehash.hex_to_hash(row['phash'])
            for other_hash, location in protected_hashes:
                distance = hashed - other_hash
                if distance <= PHASH_DISTANCE and location not in matched:
                    similarity_flags.append({'sceneId': row['id'], 'sceneRelativePath': row['relativePath'],
                        'otherKind': location['kind'], 'otherRelativePath': location['relativePath'],
                        'distance': int(distance)})
    # OOD/data cross-overlap must also be resolved in the threshold notebook.
    external_conflicts = [values for values in protected_bytes.values()
                         if len({v['kind'] for v in values}) > 1]
    if external_conflicts:
        raise ValueError('Mango/OOD exact input overlap exists; inspect folders before training: ' + repr(external_conflicts[:5]))
    return {'datasetImages': len(dataset), 'sceneOverlapExclusions': conflicts,
            'sceneCrossCollectionSimilarityFlags': similarity_flags,
            'oldOodImagesAudited': sum(r['kind'] == 'ood_old' for r in ood),
            'newOodImagesAudited': sum(r['kind'] == 'ood_new' for r in ood),
            'oldOodFolderPresent': OLD_OOD_DIR.is_dir(), 'newOodFolderPresent': NEW_OOD_DIR.is_dir(),
            'interpretation': 'Optional audit only; threshold fitting performs its own complete OOD audit.'}

### Step 5: Verify the saved split and create one new run

- **Purpose:** Copy the exact saved dataset ZIP, check all image identities, audit scenes, and establish provenance.
- **Reads:** Completed V4.1.1 files, both scene folders, optional review/masks, and optional OOD audit folders.
- **Saves:** Unique run with dataset_split.zip, source_records/, source_provenance.json, scene_inventory.json, duplicate_audit.json, and external_overlap_audit.json.
- **Check:** Read exact 1,819/391/393 totals, all seven class counts, expected versus actual scene counts, exclusion details, and the full run path.

In [ ]:
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Select a Colab GPU runtime before preparing this training run.')
if SOURCE_RUN is None or not SOURCE_CHOICE_NOTES.strip() or not BACKGROUND_NOTES.strip():
    raise ValueError('Set the exact completed V4.1.1 source path, source-choice notes, and background source/rights notes before preflight.')
if not (SOURCE_RUN/'training_complete.json').is_file():
    raise FileNotFoundError('Select the full completed V4.1.1 training run path.')
require_output_disjoint(OUTPUT_ROOT, (SOURCE_RUN, BACKGROUND_DIR, NEW_BACKGROUND_DIR, OLD_OOD_DIR, NEW_OOD_DIR))
source_complete, source_plan, source_manifest, counts = load_source_contract(SOURCE_RUN)
source_white_review, reusable_masks, reuse_audit = load_reusable_reviews(SOURCE_RUN, source_complete, source_manifest)
run_id = 'v4-3-0-training-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATASET_DIR = Path('/content/mangoleaf-v4-3-0') / run_id / 'dataset'
copy_verified(SOURCE_RUN/'dataset_split.zip', RUN_DIR/'dataset_split.zip', source_complete['datasetSplitSha256'])
DATASET_SPLIT_SHA256 = sha256_file(RUN_DIR/'dataset_split.zip')
records = extract_dataset(RUN_DIR/'dataset_split.zip', source_manifest['images'], DATASET_DIR)
for row in records:
    load_pixels(row['path'])  # Decode every split image without evaluating its class.
for identity, mask in list(reusable_masks.items()):
    row = next(r for r in records if r['sha256'] == identity and r['subset'] == 'train')
    with Image.open(row['path']) as picture:
        if picture.size != tuple(mask['maskSize']):
            raise ValueError('Requested saved mask dimensions differ for ' + row['relativePath'] +
                             '. Investigate, or explicitly set REUSE_REVIEWED_MASKS=False for fresh mask generation.')
scene_inventory = inventory_scenes(BACKGROUND_ROOTS)
ood_audit_rows = inventory_optional_ood(OLD_OOD_DIR, 'ood_old') + inventory_optional_ood(NEW_OOD_DIR, 'ood_new')
external_audit = audit_scene_overlap(records, scene_inventory['images'], ood_audit_rows)
scene_inventory['automaticExclusions'] = sum(bool(r['automaticExclusion']) for r in scene_inventory['images'])
write_json_once(RUN_DIR/'scene_inventory.json', scene_inventory)
SCENE_INVENTORY_SHA256 = sha256_file(RUN_DIR/'scene_inventory.json')
write_json_once(RUN_DIR/'external_overlap_audit.json', external_audit)
copy_verified(SOURCE_RUN/'duplicate_audit.json', RUN_DIR/'duplicate_audit.json', source_complete['artifacts']['duplicate_audit.json'])
source_files = {}
for name in ('training_complete.json', 'plan.json', 'input_manifest.json', 'white_background_review.json', 'mask_review.json'):
    path = SOURCE_RUN/name
    if name == 'training_complete.json':
        expected = SOURCE_COMPLETION_SHA256
    else:
        expected = source_complete['artifacts'].get(name)
    if path.is_file() and expected and sha256_file(path) == expected:
        copy_verified(path, RUN_DIR/'source_records'/name, expected)
        source_files[name] = expected
write_json_once(RUN_DIR/'source_provenance.json', {
    'sourceRun': str(SOURCE_RUN), 'sourceCompletionSha256': SOURCE_COMPLETION_SHA256,
    'sourceChoiceNotes': SOURCE_CHOICE_NOTES, 'sourceFiles': source_files,
    'datasetSplitCopiedByteForByte': True, 'datasetSplitSha256': DATASET_SPLIT_SHA256,
    'historicalIdentity': EXPECTED_BASELINE_IDENTITY, 'counts': counts})
plan = {'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES, 'seed': SEED,
    'splitFractions': [0.70, 0.15, 0.15], 'splitPolicy': 'reuse exact saved V4.1.1 historical membership by contents',
    'baselineIdentity': EXPECTED_BASELINE_IDENTITY, 'rawIdentity': source_plan.get('rawIdentity'),
    'phashDistance': PHASH_DISTANCE, 'batchSize': BATCH_SIZE, 'imageSize': list(IMAGE_SIZE),
    'stage1': [STAGE1_EPOCHS, STAGE1_LR], 'stage2': [STAGE2_EPOCHS, STAGE2_LR],
    'augmentation': [AUG_FLIP, AUG_ROTATION_DEG, AUG_ZOOM, AUG_SHEAR, AUG_BRIGHTNESS],
    'preprocessing': PREPROCESSING, 'confidenceThreshold': CONFIDENCE_THRESHOLD,
    'minimumValidationRetention': MIN_ID_RETENTION, 'runtime': RUNTIME,
    'evaluationStatus': EVALUATION_STATUS, 'evaluationNotes': EVALUATION_NOTES,
    'arms': list(ARMS), 'blendProbability': BLEND_PROBABILITY, 'localLightProbability': LOCAL_LIGHT_PROBABILITY,
    'lightAmplitude': list(LIGHT_AMPLITUDE), 'lightRadius': list(LIGHT_RADIUS),
    'backgroundNotes': BACKGROUND_NOTES, 'backgroundRoots': scene_inventory['roots'],
    'expectedBackgroundCounts': EXPECTED_BACKGROUND_COUNTS, 'sceneInventorySha256': SCENE_INVENTORY_SHA256,
    'sourceCompletionSha256': SOURCE_COMPLETION_SHA256,
    'sourceProvenanceSha256': sha256_file(RUN_DIR/'source_provenance.json'),
    'whiteEligibilityRule': 'every training image manually reviewed; hash-valid prior decisions are suggestions',
    'maskRule': 'reviewed train-only masks; hash-valid saved masks reused; original image when mask rejected',
    'modelSelection': 'minimum val_loss within each stage',
    'temperatureRule': 'validation NLL only; logits divided by temperature once'}
print('Unique V4.3.0 RUN_DIR:', RUN_DIR)
for subset in counts:
    print(subset, counts[subset], 'total:', sum(counts[subset].values()))
print('Expected and decoded scene counts:', scene_inventory['poolCounts'])
print('Reported total:', scene_inventory['expectedTotalFiles'], 'actual supported:', scene_inventory['supportedFiles'],
      'readable:', scene_inventory['readableImages'], 'unique decoded:', scene_inventory['uniqueDecodedImages'])
print('Automatic scene exclusions:', scene_inventory['automaticExclusions'], 'similarity flags:', len(scene_inventory['similarityFlags']))
print('Scene/mango/OOD similarity flags:', len(external_audit['sceneCrossCollectionSimilarityFlags']))
print('Hash-valid saved mask candidates:', len(reusable_masks))
print('Read duplicate_audit.json, scene_inventory.json, and external_overlap_audit.json before review.')

### Step 6: Make contact sheets for both scene pools

- **Purpose:** Show all reported scene files, including repeated and unreadable rows, for review.
- **Reads:** Frozen scene inventory and unchanged original scene bytes.
- **Saves:** scene_previews/page-*.jpg.
- **Check:** Open every page. Exclude mango subjects, disease-revealing content, text, people, unsuitable settings, and repeated views when appropriate; do not count automatic exclusions as usable scenes.

In [ ]:
SCENE_SHEET_DIR = RUN_DIR/'scene_previews'
SCENE_SHEET_DIR.mkdir(exist_ok=False)
for page_number, start in enumerate(range(0, len(scene_inventory['images']), 24), 1):
    page = Image.new('RGB', (4*280, 6*240), 'white')
    draw = ImageDraw.Draw(page)
    for index, row in enumerate(scene_inventory['images'][start:start+24]):
        x, y = (index % 4)*280, (index // 4)*240
        if row['readable']:
            with Image.open(row['path']) as picture:
                thumbnail = ImageOps.contain(picture.convert('RGB'), (272, 185))
            page.paste(thumbnail, (x+(280-thumbnail.width)//2, y))
        else:
            draw.text((x+8, y+70), 'UNREADABLE: inspect source file', fill='red')
        draw.text((x+4, y+187), row['id'] + ' ' + row['pool'], fill='black')
        draw.text((x+4, y+204), row['sourceRelativePath'][:38], fill='black')
        draw.text((x+4, y+220), (row['automaticExclusion'] or 'Review suitability')[:38], fill='red' if row['automaticExclusion'] else 'black')
    page.save(SCENE_SHEET_DIR/f'page-{page_number:03d}.jpg', quality=92)
print('Open every scene page:', SCENE_SHEET_DIR)
print('Inspect all similarity flags by ID in scene_inventory.json; exclude unsuitable or redundant views explicitly.')
flagged_scene_ids = {r['first'] for r in scene_inventory['similarityFlags']} | {r['second'] for r in scene_inventory['similarityFlags']}
flagged_scene_ids |= {r['sceneId'] for r in external_audit['sceneCrossCollectionSimilarityFlags']}
print('Review these scene IDs against scene/scene and scene/mango/OOD flags:', sorted(flagged_scene_ids))

### Step 7: Review scenes, explain counts, and freeze the usable pool

- **Purpose:** Approve suitable unique scenes and preserve their exact bytes for future training/figures.
- **Reads:** Every scene contact sheet and inventory similarity/exclusion flags.
- **Saves:** backgrounds/, scene_review.json, and final input_manifest.json.
- **Check:** Reported 200 files can become fewer approved unique scenes after explicit exclusions. Unexpected decoded/file counts need the separate acknowledgement and written explanation.

In [ ]:
SCENE_REVIEW_COMPLETE = False
EXCLUDE_SCENE_IDS = []  # IDs from the contact sheets that should not be replacement scenes.
REVIEWED_SIMILARITY_SCENE_IDS = []  # Every ID printed in Step 6, after checking its flagged pair(s).
SCENE_REVIEW_NOTES = ''  # State suitability decisions and what similarity flags were checked.
ACKNOWLEDGE_SCENE_COUNT_MISMATCH = False
SCENE_COUNT_MISMATCH_NOTES = ''  # Explain a documented count difference; never silently reset 62/138.

if not SCENE_REVIEW_COMPLETE or not SCENE_REVIEW_NOTES.strip():
    raise ValueError('Review every scene sheet, write SCENE_REVIEW_NOTES, then approve SCENE_REVIEW_COMPLETE.')
if sha256_file(RUN_DIR/'scene_inventory.json') != SCENE_INVENTORY_SHA256:
    raise RuntimeError('Scene inventory changed after preflight.')
scene_by_id = {row['id']: row for row in scene_inventory['images']}
if len(EXCLUDE_SCENE_IDS) != len(set(EXCLUDE_SCENE_IDS)) or set(EXCLUDE_SCENE_IDS) - set(scene_by_id):
    raise ValueError('Scene exclusions contain a repeated or unknown ID.')
if set(REVIEWED_SIMILARITY_SCENE_IDS) != flagged_scene_ids:
    raise ValueError('Review every flagged scene pair, then record every flagged scene ID in REVIEWED_SIMILARITY_SCENE_IDS.')
mismatch = any(row['supportedFiles'] != row['expectedFiles'] or row['readableImages'] != row['expectedFiles']
               for row in scene_inventory['poolCounts'].values())
if mismatch and (not ACKNOWLEDGE_SCENE_COUNT_MISMATCH or not SCENE_COUNT_MISMATCH_NOTES.strip()):
    raise ValueError('The decoded scene counts differ from reported 62 + 138. Inspect the inventory and explicitly explain/acknowledge the discrepancy.')
background_records, exclusions = [], []
for row in scene_inventory['images']:
    reason = row['automaticExclusion'] or ('reviewer exclusion' if row['id'] in EXCLUDE_SCENE_IDS else None)
    if reason:
        exclusions.append({'id': row['id'], 'pool': row['pool'], 'relativePath': row['relativePath'],
                           'sha256': row['sha256'], 'reason': reason, 'duplicateOf': row.get('duplicateOf')})
        continue
    if sha256_file(row['path']) != row['sha256'] or rgb_identity(row['path'])[0] != row['decodedRgbSha256']:
        raise RuntimeError('Scene changed since preview: ' + row['relativePath'])
    frozen_relative = 'backgrounds/' + row['id'] + Path(row['path']).suffix.lower()
    frozen_path = inside(RUN_DIR, frozen_relative)
    copy_verified(row['path'], frozen_path, row['sha256'])
    background_records.append({**row, 'path': str(frozen_path), 'frozenRelativePath': frozen_relative})
if not background_records or len({r['decodedRgbSha256'] for r in background_records}) != len(background_records):
    raise ValueError('Approved scene pool must contain at least one unique decoded image.')
approved_counts = dict(Counter(r['pool'] for r in background_records))
if set(approved_counts) != set(BACKGROUND_ROOTS):
    raise ValueError('Both original and fresh scene pools must retain approved content; inspect exclusions.')
write_json_once(RUN_DIR/'scene_review.json', {
    'reviewedAt': datetime.now(timezone.utc).isoformat(), 'allSceneImagesReviewed': True,
    'sceneInventorySha256': SCENE_INVENTORY_SHA256, 'sourceNotes': BACKGROUND_NOTES,
    'reviewNotes': SCENE_REVIEW_NOTES, 'expectedCounts': EXPECTED_BACKGROUND_COUNTS,
    'reviewedSimilaritySceneIds': sorted(REVIEWED_SIMILARITY_SCENE_IDS),
    'externalOverlapAuditSha256': sha256_file(RUN_DIR/'external_overlap_audit.json'),
    'actualCounts': scene_inventory['poolCounts'], 'countMismatchAcknowledged': bool(mismatch and ACKNOWLEDGE_SCENE_COUNT_MISMATCH),
    'countMismatchNotes': SCENE_COUNT_MISMATCH_NOTES if mismatch else None,
    'excludedImages': exclusions, 'approvedIds': [r['id'] for r in background_records],
    'approvedUniqueCounts': approved_counts, 'approvedUniqueTotal': len(background_records)})
SCENE_REVIEW_SHA256 = sha256_file(RUN_DIR/'scene_review.json')
manifest = {'classes': CLASS_NAMES, 'counts': counts,
    'images': [{k: v for k, v in r.items() if k != 'path'} for r in records],
    'backgroundImages': background_records, 'oodAuditImages': ood_audit_rows,
    'externalOverlapAudit': external_audit}
write_json_once(RUN_DIR/'input_manifest.json', manifest)
MANIFEST_SHA256 = sha256_file(RUN_DIR/'input_manifest.json')
plan.update({'backgroundCount': len(background_records), 'approvedBackgroundCounts': approved_counts,
             'sceneReviewSha256': SCENE_REVIEW_SHA256})
print('Reported input pool: 62 + 138 = 200; actual supported/readable/unique:',
      scene_inventory['supportedFiles'], scene_inventory['readableImages'], scene_inventory['uniqueDecodedImages'])
print('Exclusions:', len(exclusions), 'final approved unique scenes:', len(background_records), approved_counts)
print('Saved explicit scene review and frozen input manifest:', RUN_DIR)

### Step 8: Make white-background sheets for every training image

- **Purpose:** Review all 1,819 training foregrounds rather than assuming more scenes means more usable masks.
- **Reads:** Frozen training split only and any hash-valid source white decisions.
- **Saves:** white_background_previews/page-*.jpg and tentative suggestions.
- **Check:** Open every page, including unmarked pale/off-white backgrounds; leave natural-scene foregrounds out. Validation and test are ineligible.

In [ ]:
def image_id(row):
    # Stable ID links preview rows, review records, and masks despite long filenames.
    return hashlib.sha256(row['relativePath'].encode('utf-8')).hexdigest()[:20]


def stable_seed(*parts):
    return int.from_bytes(hashlib.sha256('|'.join(map(str, parts)).encode()).digest()[:8], 'big')


def white_border_score(path):
    # Mean near-white border coverage is a suggestion, not a segmentation decision.
    with Image.open(path) as picture:
        sample = np.asarray(picture.convert('RGB').resize((128, 128)), dtype=np.uint8)
    width = 12
    border = np.concatenate([sample[:width].reshape(-1, 3), sample[-width:].reshape(-1, 3),
                             sample[:, :width].reshape(-1, 3), sample[:, -width:].reshape(-1, 3)])
    return float(np.mean((border.min(axis=1) >= 220) & ((border.max(axis=1) - border.min(axis=1)) <= 25)))


train_records = [r for r in records if r['subset'] == 'train']
train_by_id = {image_id(r): r for r in train_records}
if len(train_by_id) != len(train_records):
    raise ValueError('Training review IDs are not unique.')
source_eligible_paths = set(source_white_review['eligibleRelativePaths']) if source_white_review else set()
prior_white_ids = {key for key, row in train_by_id.items() if row['relativePath'] in source_eligible_paths}
white_suggestions = {key: white_border_score(row['path']) >= 0.70 for key, row in train_by_id.items()}
WHITE_SHEET_DIR = RUN_DIR / 'white_background_previews'
WHITE_SHEET_DIR.mkdir(exist_ok=False)
for page_index, start in enumerate(range(0, len(train_records), 36), 1):
    page = Image.new('RGB', (6 * 190, 6 * 170), 'white')
    draw = ImageDraw.Draw(page)
    for offset, row in enumerate(train_records[start:start + 36]):
        key = image_id(row)
        with Image.open(row['path']) as picture:
            thumb = ImageOps.contain(picture.convert('RGB'), (184, 140))
        x, y = (offset % 6) * 190, (offset // 6) * 170
        page.paste(thumb, (x + (190 - thumb.width) // 2, y))
        draw.text((x + 2, y + 142), key, fill='black')
        draw.text((x + 2, y + 155), row['label'][:17] + (' W?' if white_suggestions[key] else '') + (' P' if key in prior_white_ids else ''), fill='black')
    page.save(WHITE_SHEET_DIR / f'page-{page_index:03d}.jpg', quality=90)
print('Review all', page_index, 'pages:', WHITE_SHEET_DIR)
print('Automatic white suggestions:', sum(white_suggestions.values()), 'of', len(train_records))

# Hash-verified prior decisions help locate earlier choices, but are not new approval.
print('Prior hash-valid white decisions:', len(prior_white_ids), '(review every current page again)')
print('W? is a new heuristic suggestion; P is a verified prior decision, still requiring current review.')

### Step 9: Approve the current white-background foregrounds

- **Purpose:** Turn all-page review into an explicit train-only eligibility list.
- **Reads:** Every white page, prior suggestions, and edited ADD_WHITE_IDS/REMOVE_WHITE_IDS.
- **Saves:** white_background_review.json with approved image identities and per-class counts.
- **Check:** Approve only after reviewing all pages. Imported suggestions and border heuristics may both be wrong; record additions/removals.

In [ ]:
WHITE_REVIEW_COMPLETE = False
ADD_WHITE_IDS = []     # IDs that the border heuristic missed.
REMOVE_WHITE_IDS = []  # Suggested or imported IDs whose backgrounds are not clean white.

if not WHITE_REVIEW_COMPLETE:
    raise ValueError('Inspect every white-background page, edit ID lists, then set WHITE_REVIEW_COMPLETE = True.')
if set(ADD_WHITE_IDS) & set(REMOVE_WHITE_IDS):
    raise ValueError('An ID cannot be both added and removed.')
if (set(ADD_WHITE_IDS) | set(REMOVE_WHITE_IDS)) - set(train_by_id):
    raise ValueError('Unknown white-background review ID.')
eligible_ids = ({key for key, suggested in white_suggestions.items() if suggested} |
                prior_white_ids | set(ADD_WHITE_IDS)) - set(REMOVE_WHITE_IDS)
if not eligible_ids:
    raise ValueError('No white-background training images approved.')
WHITE_REVIEW_PATH = RUN_DIR / 'white_background_review.json'
write_json_once(WHITE_REVIEW_PATH, {
    'reviewedAt': datetime.now(timezone.utc).isoformat(), 'allTrainImagesReviewed': True,
    'inputManifestSha256': MANIFEST_SHA256, 'suggestionThreshold': 0.70,
    'sourceWhiteReviewSha256': source_complete.get('whiteReviewSha256') if source_white_review else None,
    'importedSuggestionIds': sorted(prior_white_ids),
    'eligibleImageIdentities': {key: train_by_id[key]['sha256'] for key in sorted(eligible_ids)},
    'addIds': sorted(ADD_WHITE_IDS), 'removeIds': sorted(REMOVE_WHITE_IDS),
    'eligibleIds': sorted(eligible_ids), 'eligibleRelativePaths':
        sorted(train_by_id[key]['relativePath'] for key in eligible_ids)})
WHITE_REVIEW_SHA256 = sha256_file(WHITE_REVIEW_PATH)
print('Approved white-background images:', len(eligible_ids), 'of', len(train_records))
eligible_counts = Counter(train_by_id[key]['label'] for key in eligible_ids)
print('Approved per class:', {label: eligible_counts[label] for label in CLASS_NAMES})

### Step 10: Reuse verified masks and generate missing candidates

- **Purpose:** Preserve valid reviewed masks by image/mask hashes and segment newly eligible/absent cases, or generate fresh candidates after the explicit reuse=False choice.
- **Reads:** Current approved training IDs, source mask/review hashes, and training-image bytes.
- **Saves:** masks/*.png, mask_jobs.json, mask_reuse_audit.json, and mask_requirements.txt.
- **Check:** Reuse is evidence-based, not filename-based. Changed requested masks never silently fall back. Every reused and newly generated mask still needs a current composite review.

In [ ]:
import venv
mask_dir = RUN_DIR/'masks'
mask_dir.mkdir(exist_ok=False)
jobs = []
for key in sorted(eligible_ids):
    row = train_by_id[key]
    saved = reusable_masks.get(row['sha256']) if REUSE_REVIEWED_MASKS else None
    job = {**row, 'id': key, 'maskPath': str(mask_dir/(key+'.png')), 'reused': False}
    if saved is not None:
        if saved['id'] != key or saved['relativePath'] != row['relativePath']:
            raise ValueError('Reusable mask ID/path does not match current historical membership.')
        copy_verified(saved['sourceMaskPath'], job['maskPath'], saved['maskSha256'])
        job.update({'reused': True, 'sourceMaskSha256': saved['maskSha256'], 'sourceRun': str(SOURCE_RUN)})
    jobs.append(job)
worker_input = RUN_DIR/'mask_jobs.json'
write_json_once(worker_input, jobs)
fresh_jobs = [job for job in jobs if not job['reused']]
reuse_audit.update({'reuseEnabled': REUSE_REVIEWED_MASKS, 'currentEligibleImages': len(jobs),
    'reusedCount': len(jobs)-len(fresh_jobs), 'generatedCount': len(fresh_jobs),
    'reusedImages': [{'id': j['id'], 'imageSha256': j['sha256'], 'maskSha256': j['sourceMaskSha256']}
                     for j in jobs if j['reused']]})
write_json_once(RUN_DIR/'mask_reuse_audit.json', reuse_audit)
# Install segmentation only when needed, inside its own CPU environment.
# Keeping it isolated avoids changing TensorFlow, NumPy, or the model-training runtime.
MASK_ENV = Path('/content/mangoleaf-mask-env-v4-3-0')
MASK_PYTHON = MASK_ENV/'bin/python'
if fresh_jobs:
    venv.EnvBuilder(with_pip=False).create(MASK_ENV)
    subprocess.run([sys.executable, '-m', 'pip', '--python', str(MASK_PYTHON),
                   'install', '--quiet', 'pip==25.2', 'rembg[cpu]==2.0.67'], check=True)
worker = r"""
import json, sys, hashlib, subprocess
from pathlib import Path
from PIL import Image
from rembg import remove, new_session
jobs = [job for job in json.loads(Path(sys.argv[1]).read_text()) if not job['reused']]
session = new_session('u2net', providers=['CPUExecutionProvider'])
for i, job in enumerate(jobs):
    raw = Path(job['path']).read_bytes()
    if hashlib.sha256(raw).hexdigest() != job['sha256']:
        raise RuntimeError('Training image changed: ' + job['path'])
    with Image.open(job['path']) as source:
        picture = source.convert('RGB')
        picture.info.clear()  # Match v3 decoding; prevent automatic EXIF rotation.
        mask = remove(picture, session=session, only_mask=True,
                      post_process_mask=False, alpha_matting=False).convert('L')
        if mask.size != picture.size:
            raise RuntimeError('Mask dimensions differ from source: ' + job['path'])
        mask.save(job['maskPath'])
    if (i + 1) % 50 == 0: print('Masks:', i + 1, '/', len(jobs), flush=True)
Path(sys.argv[2]).write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
"""
if fresh_jobs:
    subprocess.run([str(MASK_PYTHON), '-c', worker, str(worker_input), str(RUN_DIR / 'mask_requirements.txt')], check=True)
else:
    (RUN_DIR/'mask_requirements.txt').write_text('All eligible masks were reused by verified image and mask hashes; no segmentation environment executed.\n', encoding='utf-8')
print('Reused masks:', len(jobs)-len(fresh_jobs), 'fresh candidates:', len(fresh_jobs))

### Step 11: Inspect all foreground masks and composites

- **Purpose:** Show whether masks preserve lesions and leaf/fruit edges while removing only the background.
- **Reads:** Every eligible train image, candidate mask, and approved frozen scene pool.
- **Saves:** mask_previews/page-*.jpg and mask_inventory.json.
- **Check:** Inspect every original/mask/composite row, including reused masks. Reject lost disease marks, chopped edges, large white halos, or unreliable segmentation.

In [ ]:
def preview_composite(foreground, mask, background):
    # Keep object and lesions; replace only pixels marked as scene by the mask.
    foreground = foreground.convert('RGB')
    mask = mask.convert('L').resize(foreground.size, Image.Resampling.BILINEAR)
    background = ImageOps.fit(background.convert('RGB'), foreground.size,
                              method=Image.Resampling.BILINEAR)
    return Image.composite(foreground, background, mask)


MASK_PREVIEW_DIR = RUN_DIR / 'mask_previews'
MASK_PREVIEW_DIR.mkdir(exist_ok=False)
mask_rows = []
for job in jobs:
    with Image.open(job['maskPath']) as mask:
        hist = mask.convert('L').histogram()
        area = sum(hist[128:]) / sum(hist)
    mask_rows.append({**job, 'maskSha256': sha256_file(job['maskPath']),
                      'foregroundFraction': area, 'areaUsable': 0.05 <= area <= 0.98})
for start in range(0, len(mask_rows), 8):
    page = Image.new('RGB', (672, 8 * 252), 'white')
    draw = ImageDraw.Draw(page)
    for j, row in enumerate(mask_rows[start:start + 8]):
        with Image.open(row['path']) as source, Image.open(row['maskPath']) as mask, \
             Image.open(background_records[(start+j) % len(background_records)]['path']) as scene:
            original = source.convert('RGB').resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
            small_mask = mask.convert('L').resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
            combined = preview_composite(original, small_mask, scene)
            for k, picture in enumerate((original, small_mask.convert('RGB'), combined)):
                page.paste(picture, (224*k, j*252+28))
            draw.text((2, j*252), row['id'] + ' ' + row['label'] +
                      (' [AUTO-SKIP]' if not row['areaUsable'] else ''), fill='black')
    page.save(MASK_PREVIEW_DIR / f'page-{start//8+1:03d}.jpg', quality=90)
write_json_once(RUN_DIR / 'mask_inventory.json', mask_rows)
print('Review all', math.ceil(len(mask_rows)/8), 'pages:', MASK_PREVIEW_DIR)
print('Automatic area exclusions:', sum(not row['areaUsable'] for row in mask_rows))

### Step 12: Approve or reject masks for this run

- **Purpose:** Freeze the approved train-only masks; rejected foregrounds retain their original presentation.
- **Reads:** Every mask-composite page and edited REJECT_MASK_IDS.
- **Saves:** mask_review.json with image/mask hashes and per-class approved counts.
- **Check:** The 30% replacement probability applies among approved masks only. Read the expected fraction of all training presentations receiving scenes.

In [ ]:
MASK_REVIEW_COMPLETE = False
REJECT_MASK_IDS = []  # Copy IDs from bad preview rows; originals remain in training.

if not MASK_REVIEW_COMPLETE:
    raise ValueError('Review every mask preview; set MASK_REVIEW_COMPLETE = True only afterward.')
if set(REJECT_MASK_IDS) - {row['id'] for row in mask_rows}:
    raise ValueError('Unknown rejected mask ID.')
approved_rows = []
for row in mask_rows:
    if sha256_file(row['maskPath']) != row['maskSha256']:
        raise RuntimeError('Mask changed after preview: ' + row['id'])
    approved_rows.append({**row, 'useForBlend': bool(row['areaUsable'] and row['id'] not in REJECT_MASK_IDS)})
approved_masks = {row['relativePath']: row for row in approved_rows if row['useForBlend']}
if not approved_masks:
    raise ValueError('No usable masks remain; combined training would have no scene replacements.')
MASK_REVIEW_PATH = RUN_DIR / 'mask_review.json'
write_json_once(MASK_REVIEW_PATH, {'reviewedAt': datetime.now(timezone.utc).isoformat(),
    'whiteReviewSha256': WHITE_REVIEW_SHA256, 'allMasksReviewed': True,
    'maskReuseAuditSha256': sha256_file(RUN_DIR/'mask_reuse_audit.json'), 'rejectedIds': sorted(REJECT_MASK_IDS),
    'masks': approved_rows})
MASK_REVIEW_SHA256 = sha256_file(MASK_REVIEW_PATH)
print('Approved masks per class:', dict(Counter(row['label'] for row in approved_masks.values())))
print('Expected fraction of all training presentations with a scene:',
      BLEND_PROBABILITY * len(approved_masks) / len(train_records))

### Step 13: Preview localized highlights and shadows

- **Purpose:** Check modest spatial lighting variation without destroying lesions or object boundaries.
- **Reads:** Four training examples per each of the seven classes.
- **Saves:** lighting_previews/<class>.jpg.
- **Check:** Review all seven pages. The sign, radius, amplitude, and clipping are the unchanged V4.1.1 local-lighting policy.

In [ ]:
def add_local_light(pixels, key, epoch, force=None):
    rng = random.Random(stable_seed(SEED, key, epoch, 'local-light'))
    if force is None and rng.random() >= LOCAL_LIGHT_PROBABILITY:
        return pixels
    sign = force if force is not None else rng.choice((-1, 1))
    amplitude = rng.uniform(*LIGHT_AMPLITUDE)
    radius = rng.uniform(*LIGHT_RADIUS) * min(IMAGE_SIZE)
    cx, cy = rng.uniform(0, IMAGE_SIZE[1]-1), rng.uniform(0, IMAGE_SIZE[0]-1)
    yy, xx = np.mgrid[:IMAGE_SIZE[0], :IMAGE_SIZE[1]]
    field = np.exp(-((xx-cx)**2 + (yy-cy)**2) / (2*radius**2))[..., None]
    return np.clip(pixels + sign * amplitude * field, 0, 255).astype(np.float32)


LIGHT_PREVIEW_DIR = RUN_DIR / 'lighting_previews'
LIGHT_PREVIEW_DIR.mkdir(exist_ok=False)
for label in CLASS_NAMES:
    sample_rows = [row for row in train_records if row['label'] == label][:4]
    page = Image.new('RGB', (3*224, len(sample_rows)*252), 'white')
    draw = ImageDraw.Draw(page)
    for index, row in enumerate(sample_rows):
        original = load_pixels(row['path']).numpy()
        for column, values in enumerate((original,
                add_local_light(original, row['relativePath'], 0, force=1),
                add_local_light(original, row['relativePath'], 0, force=-1))):
            page.paste(Image.fromarray(np.uint8(np.clip(values, 0, 255))),
                       (column*224, index*252+28))
        draw.text((2, index*252), row['label'] + ': original | highlight | shadow', fill='black')
    page.save(LIGHT_PREVIEW_DIR / f'{label}.jpg', quality=95)
print('Review all seven class pages in:', LIGHT_PREVIEW_DIR)

### Step 14: Approve the local-lighting policy

- **Purpose:** Freeze reviewed lighting settings and their preview hashes.
- **Reads:** All seven lighting previews.
- **Saves:** lighting_review.json.
- **Check:** Approve only after inspecting lesions, highlights, shadows, and clipping. Local lighting applies independently to 30% of all training presentations.

In [ ]:
LIGHT_REVIEW_COMPLETE = False
if not LIGHT_REVIEW_COMPLETE:
    raise ValueError('Inspect local highlight/shadow previews, then set LIGHT_REVIEW_COMPLETE = True.')
LIGHT_REVIEW_PATH = RUN_DIR / 'lighting_review.json'
write_json_once(LIGHT_REVIEW_PATH, {'reviewedAt': datetime.now(timezone.utc).isoformat(),
    'allClassesReviewed': True, 'probability': LOCAL_LIGHT_PROBABILITY,
    'amplitude': list(LIGHT_AMPLITUDE), 'radius': list(LIGHT_RADIUS),
    'previewSha256': {label: sha256_file(LIGHT_PREVIEW_DIR/f'{label}.jpg') for label in CLASS_NAMES}})
LIGHT_REVIEW_SHA256 = sha256_file(LIGHT_REVIEW_PATH)

### Step 15: Define the unchanged base augmentation

- **Purpose:** Keep horizontal/vertical flip, rotation, zoom, shear, and global brightness from V4.1.1.
- **Reads:** Frozen geometric and brightness settings.
- **Saves:** Keras augmentation builder only.
- **Check:** Rotation remains 30/360 of a full turn; shear uses the same bilinear transform; no extra augmentation policy is introduced.

In [ ]:
# These random transforms remain training-only and retain the V4.1.1 recipe.
@tf.keras.utils.register_keras_serializable(package="mangoleaf")
class RandomShear(tf.keras.layers.Layer):
    # Random shear about the image centre. Keras ships no version-stable equivalent.
    def __init__(self, factor=0.2, fill_mode="NEAREST", **kwargs):
        super().__init__(**kwargs)
        self.factor = factor
        self.fill_mode = fill_mode

    def call(self, inputs, training=False):
        if not training:
            return inputs
        shape = tf.shape(inputs)
        n = shape[0]
        h = tf.cast(shape[1], tf.float32)
        w = tf.cast(shape[2], tf.float32)
        cx, cy = w / 2.0, h / 2.0

        sx = tf.random.uniform([n], -self.factor, self.factor)
        sy = tf.random.uniform([n], -self.factor, self.factor)
        one  = tf.ones_like(sx)
        zero = tf.zeros_like(sx)

        # Output-to-input map. a2 and b2 cancel the centre offset the shear introduces.
        transforms = tf.stack([one, sx, -sx * cy,
                               sy, one, -sy * cx,
                               zero, zero], axis=1)

        return tf.raw_ops.ImageProjectiveTransformV3(
            images=inputs,
            transforms=transforms,
            output_shape=shape[1:3],
            fill_value=0.0,
            interpolation="BILINEAR",
            fill_mode=self.fill_mode)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"factor": self.factor, "fill_mode": self.fill_mode})
        return cfg


def make_augmentation():
    return tf.keras.Sequential([
    tf.keras.layers.RandomFlip(AUG_FLIP, seed=SEED),
    tf.keras.layers.RandomRotation(AUG_ROTATION_DEG / 360.0,
                                   fill_mode="nearest", seed=SEED),
    tf.keras.layers.RandomZoom(AUG_ZOOM, fill_mode="nearest", seed=SEED),
    RandomShear(AUG_SHEAR),
    tf.keras.layers.RandomBrightness(AUG_BRIGHTNESS, value_range=(0.0, 255.0), seed=SEED),
], name="augmentation")

### Step 16: Define the combined loader, model, and calibration

- **Purpose:** Connect reviewed scenes/local lighting to the original backbone/head and validation-only temperature fit.
- **Reads:** Frozen train rows/masks/scenes, validation rows, and base augmentation.
- **Saves:** Loader/model/calibration helpers only.
- **Check:** Only combined is valid; train-only augmentation is enforced. Validation has no augmentation, and test/OOD cannot enter these helpers.

In [ ]:
def choose_background(key, epoch):
    rng = random.Random(stable_seed(SEED, key, epoch, 'background'))
    if rng.random() >= BLEND_PROBABILITY:
        return None
    return rng.choice(background_records)


def load_training_sample(row, arm, epoch):
    if arm != 'combined' or row['subset'] != 'train':
        raise ValueError('Only combined training may augment training images.')
    pixels = load_pixels(row['path']).numpy().astype(np.float32)
    key = row['relativePath']
    if key in approved_masks:
        scene = choose_background(key, epoch)
        expected_scene = background_schedule_lookup.get((epoch, key))
        if (scene['sha256'] if scene else None) != expected_scene:
            raise RuntimeError('Scene choice differs from the frozen schedule.')
        if scene is not None:
            mask_row = approved_masks[key]
            if sha256_file(mask_row['maskPath']) != mask_row['maskSha256']:
                raise RuntimeError('Approved mask changed: ' + mask_row['id'])
            with Image.open(mask_row['maskPath']) as picture:
                alpha = np.asarray(picture.convert('L').resize(IMAGE_SIZE,
                    Image.Resampling.BILINEAR), dtype=np.float32)[..., None] / 255.0
            with Image.open(scene['path']) as picture:
                background = np.asarray(ImageOps.fit(picture.convert('RGB'), IMAGE_SIZE,
                    method=Image.Resampling.BILINEAR), dtype=np.float32)
            pixels = pixels*alpha + background*(1.0-alpha)
    # Scene and local-light decisions are independent; then Keras base transforms run.
    pixels = add_local_light(pixels, key, epoch)
    return pixels.astype(np.float32), CLASS_NAMES.index(row['label'])


def make_dataset(rows, arm=None, epoch=0, augmentation=None):
    if arm is not None and (arm != 'combined' or any(r['subset'] != 'train' for r in rows)):
        raise ValueError('Augmented model.fit input must contain training images only.')
    if arm is None and any(r['subset'] != 'val' for r in rows):
        raise ValueError('Unaugmented training/calibration dataset must contain validation images only.')
    def generate():
        order = list(range(len(rows)))
        if arm is not None:
            random.Random(stable_seed(SEED, epoch, 'order')).shuffle(order)
        for index in order:
            row = rows[index]
            if arm is None:
                yield load_pixels(row['path']).numpy().astype(np.float32), CLASS_NAMES.index(row['label'])
            else:
                yield load_training_sample(row, arm, epoch)
    dataset = tf.data.Dataset.from_generator(generate, output_signature=(
        tf.TensorSpec(IMAGE_SIZE + (3,), tf.float32), tf.TensorSpec((), tf.int32)))
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.apply(tf.data.experimental.assert_cardinality(math.ceil(len(rows)/BATCH_SIZE)))
    if augmentation is not None:
        dataset = dataset.map(lambda x,y: (augmentation(x, training=True), y), num_parallel_calls=1)
    return dataset.prefetch(1)


def make_model():
    base = tf.keras.applications.MobileNetV3Large(input_shape=IMAGE_SIZE + (3,),
        include_top=False, weights='imagenet', include_preprocessing=True)
    # The first stage learns only the new seven-class head from ImageNet features.
    base.trainable = False
    inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,), name='input_image')
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = tf.keras.layers.Dropout(0.2, name='dropout')(x)
    logits = tf.keras.layers.Dense(len(CLASS_NAMES), name='logits')(x)
    probs = tf.keras.layers.Activation('softmax', dtype='float32', name='probs')(logits)
    return tf.keras.Model(inputs, probs, name='mangoleaf_v4_3_0'), base


def calibrate_arm(model, val_rows):
    from scipy.optimize import minimize_scalar
    logits_model = tf.keras.Model(model.inputs, model.get_layer('logits').output)
    logits = logits_model.predict(make_dataset(val_rows), verbose=0).astype(np.float64)
    labels = np.array([CLASS_NAMES.index(r['label']) for r in val_rows])
    def nll(temperature):
        z = logits / temperature
        z -= z.max(axis=1, keepdims=True)
        return float((np.log(np.exp(z).sum(axis=1)) - z[np.arange(len(labels)), labels]).mean())
    # Calibration uses validation NLL; do not fit temperature on test/OOD reports.
    result = minimize_scalar(nll, bounds=(0.25, 20.0), method='bounded')
    if not result.success or not math.isfinite(float(result.fun)):
        raise RuntimeError('Temperature fit failed; this arm is incomplete.')
    temperature = float(result.x) if result.fun < nll(1.0) else 1.0
    before = predict_records(model, val_rows)
    layer = model.get_layer('logits')
    kernel, bias = layer.get_weights()
    # Fold temperature into logits once so Keras and browser exports agree.
    layer.set_weights([kernel/temperature, bias/temperature])
    after = predict_records(model, val_rows)
    if not np.array_equal(before.argmax(axis=1), after.argmax(axis=1)):
        raise RuntimeError('Temperature folding changed a validation class prediction.')
    return temperature, {'before': nll(1.0), 'after': nll(temperature)}, after

### Step 17: Freeze the 15-epoch scene schedule and final plan

- **Purpose:** Record actual deterministic scene choices and all reviewed inputs before fitting.
- **Reads:** Approved masks, unique scenes, all reviews, manifest, and training settings.
- **Saves:** background_schedule.json, plan.json, and recovery_state.json with complete input hashes.
- **Check:** Read white/mask/scene counts per class and sampled scene presentations. Changed settings/inputs require a new unique run.

In [ ]:
# Freeze the actual scene choices for all 15 epochs before fitting the combined model.
# The larger reviewed scene pool changes only the scenes sampled by this recipe.
background_schedule = []
for epoch in range(STAGE1_EPOCHS + STAGE2_EPOCHS):
    for row in train_records:
        if row['relativePath'] not in approved_masks:
            continue
        scene = choose_background(row['relativePath'], epoch)
        if scene is not None:
            background_schedule.append({'epoch': epoch, 'trainRelativePath': row['relativePath'],
                'sceneRelativePath': scene['relativePath'], 'sceneSha256': scene['sha256'],
                'label': row['label']})
SCHEDULE_PATH = RUN_DIR/'background_schedule.json'
write_json_once(SCHEDULE_PATH, {'rule': 'one deterministic scene choice per approved training image and epoch for combined',
    'seed': SEED, 'presentations': background_schedule})
SCHEDULE_SHA256 = sha256_file(SCHEDULE_PATH)
background_schedule_lookup = {(row['epoch'], row['trainRelativePath']): row['sceneSha256']
                              for row in background_schedule}
plan['sceneReviewSha256'] = SCENE_REVIEW_SHA256
plan['inputManifestSha256'] = MANIFEST_SHA256
plan['datasetSplitSha256'] = DATASET_SPLIT_SHA256
plan['whiteReviewSha256'] = WHITE_REVIEW_SHA256
plan['maskReviewSha256'] = MASK_REVIEW_SHA256
plan['lightingReviewSha256'] = LIGHT_REVIEW_SHA256
plan['backgroundScheduleSha256'] = SCHEDULE_SHA256
train_counts = Counter(row['label'] for row in train_records)
mask_counts = Counter(row['label'] for row in approved_masks.values())
scene_counts = Counter(row['label'] for row in background_schedule)
plan['trainingImagesByClass'] = {label: train_counts[label] for label in CLASS_NAMES}
plan['eligibleWhiteImagesByClass'] = {label: eligible_counts[label] for label in CLASS_NAMES}
plan['approvedMaskImagesByClass'] = {label: mask_counts[label] for label in CLASS_NAMES}
plan['scenePresentationsByClass'] = {label: scene_counts[label] for label in CLASS_NAMES}
plan['expectedReplacedTrainingFraction'] = BLEND_PROBABILITY * len(approved_masks)/len(train_records)
PLAN_PATH = RUN_DIR/'plan.json'
write_json_once(PLAN_PATH, plan)
PLAN_SHA256 = sha256_file(PLAN_PATH)
print('Approved white images by class:', plan['eligibleWhiteImagesByClass'])
print('Usable masks by class:', plan['approvedMaskImagesByClass'])
print('Scene presentations over 15 epochs:', len(background_schedule))
print('Final frozen plan:', PLAN_PATH)

# This small ledger supports export recovery without trusting the current notebook state.
recovery_names = ['plan.json', 'input_manifest.json', 'dataset_split.zip', 'duplicate_audit.json',
    'external_overlap_audit.json', 'source_provenance.json', 'scene_inventory.json', 'scene_review.json',
    'white_background_review.json', 'mask_review.json', 'mask_inventory.json', 'mask_jobs.json',
    'mask_reuse_audit.json', 'lighting_review.json', 'background_schedule.json', 'mask_requirements.txt']
recovery_names += ['masks/'+r['id']+'.png' for r in mask_rows]
recovery_names += [r['frozenRelativePath'] for r in background_records]
recovery_names += [(p.relative_to(RUN_DIR)).as_posix() for p in sorted((RUN_DIR/'source_records').glob('*.json'))]
write_json_once(RUN_DIR/'recovery_state.json', {'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES,
    'armOrder': list(ARMS), 'planSha256': PLAN_SHA256,
    'artifacts': {name: sha256_file(inside(RUN_DIR, name)) for name in recovery_names}})
RECOVERY_STATE_SHA256 = sha256_file(RUN_DIR/'recovery_state.json')

### Step 18: Verify frozen inputs and calculate class weights

- **Purpose:** Fail early on changed inputs/settings and balance loss with training-only label frequencies.
- **Reads:** Final plan, recovery ledger, every image/mask/scene hash, and training labels.
- **Saves:** Class weights and checked train/validation rows in memory.
- **Check:** Confirm combined only, all seven classes, train 1,819, validation 391, and unchanged 5 + 10 recipe. No test prediction occurs.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

if (RUN_DIR/'training_complete.json').exists():
    raise RuntimeError('This run is complete. Use saved diagnostics or threshold fitting; do not retrain it.')
verify_artifacts(RUN_DIR, read_json(RUN_DIR/'recovery_state.json')['artifacts'])

if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Select a Colab GPU runtime before training.')
for path, expected in ((RUN_DIR/'plan.json', PLAN_SHA256),
                       (RUN_DIR/'input_manifest.json', MANIFEST_SHA256),
                       (RUN_DIR/'dataset_split.zip', DATASET_SPLIT_SHA256),
                       (WHITE_REVIEW_PATH, WHITE_REVIEW_SHA256),
                       (MASK_REVIEW_PATH, MASK_REVIEW_SHA256),
                       (LIGHT_REVIEW_PATH, LIGHT_REVIEW_SHA256),
                       (SCHEDULE_PATH, SCHEDULE_SHA256),
                       (RUN_DIR/'scene_review.json', SCENE_REVIEW_SHA256)):
    if sha256_file(path) != expected:
        raise RuntimeError('Frozen run artifact changed: ' + str(path))
plan = json.loads((RUN_DIR/'plan.json').read_text(encoding='utf-8'))
settings = {'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES, 'arms': list(ARMS),
    'seed': SEED, 'batchSize': BATCH_SIZE, 'imageSize': list(IMAGE_SIZE),
    'stage1': [STAGE1_EPOCHS, STAGE1_LR], 'stage2': [STAGE2_EPOCHS, STAGE2_LR],
    'blendProbability': BLEND_PROBABILITY, 'localLightProbability': LOCAL_LIGHT_PROBABILITY,
    'lightAmplitude': list(LIGHT_AMPLITUDE), 'lightRadius': list(LIGHT_RADIUS),
    'augmentation': [AUG_FLIP, AUG_ROTATION_DEG, AUG_ZOOM, AUG_SHEAR, AUG_BRIGHTNESS]}
if any(plan[k] != value for k, value in settings.items()):
    raise RuntimeError('Training settings changed after run creation. Start a new run.')
verify_records_unchanged(records)
for scene in background_records:
    if sha256_file(scene['path']) != scene['sha256']:
        raise RuntimeError('Background scene changed after run creation.')
train_rows = [r for r in records if r['subset'] == 'train']
val_rows = [r for r in records if r['subset'] == 'val']
train_labels = np.array([CLASS_NAMES.index(r['label']) for r in train_rows])
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train_labels)
class_weight = {index: float(weight) for index, weight in enumerate(weights)}
# Balanced weights use training labels only; validation/test distributions do not influence fit.
print('Class weights:', {CLASS_NAMES[i]: class_weight[i] for i in class_weight})
print('Validation order:', len(val_rows), val_rows[0]['relativePath'], '...', val_rows[-1]['relativePath'])

### Step 19: Train, select checkpoints, and calibrate one model

- **Purpose:** Train the seven-class combined model with the unchanged two stages and validation-only calibration.
- **Reads:** Frozen training/validation inputs, class weights, ImageNet initialization, and reviewed augmentation plan.
- **Saves:** arms/combined/stage{1,2}_best.weights.h5, training_history.json, model_calibrated.keras, validation_probabilities.npy, and resume_info.json.
- **Check:** An incomplete fit is not resumable here; start a new unique run. A fully saved arm is reused only by matching hashes. Reloaded validation probabilities must match, with 391 rows and seven columns.

In [ ]:
# A completed run must never be retrained by rerunning this cell.
if (RUN_DIR/'training_complete.json').exists():
    raise RuntimeError('Training is complete; go to saved diagnostics or the threshold notebook.')
ARM_ROOT = RUN_DIR / 'arms'
ARM_ROOT.mkdir(exist_ok=True)
arm_results = {}
initial_digest = None
for arm in ARMS:
    arm_dir = ARM_ROOT / arm
    if arm_dir.exists():
        info_path = arm_dir / 'resume_info.json'
        if not info_path.is_file():
            raise RuntimeError(f'{arm} has an incomplete folder; start a new run.')
        info = json.loads(info_path.read_text(encoding='utf-8'))
        if (info['arm'] != arm or info['inputManifestSha256'] != MANIFEST_SHA256 or
            info['planSha256'] != PLAN_SHA256 or info['whiteReviewSha256'] != WHITE_REVIEW_SHA256 or
            info['maskReviewSha256'] != MASK_REVIEW_SHA256 or
            info['lightingReviewSha256'] != LIGHT_REVIEW_SHA256 or
            info['backgroundScheduleSha256'] != SCHEDULE_SHA256 or
            info['sceneReviewSha256'] != SCENE_REVIEW_SHA256 or
            info['datasetSplitSha256'] != DATASET_SPLIT_SHA256 or
            info['classes'] != CLASS_NAMES or not info['temperatureFoldedIntoWeights'] or
            sha256_file(arm_dir/'model_calibrated.keras') != info['modelSha256'] or
            sha256_file(arm_dir/'validation_probabilities.npy') != info['validationProbabilitiesSha256'] or
            sha256_file(arm_dir/'training_history.json') != info['trainingHistorySha256'] or
            info['recoveryStateSha256'] != RECOVERY_STATE_SHA256):
            raise RuntimeError(f'{arm} saved artifacts changed; start a new run.')
        if initial_digest is None:
            initial_digest = info['initialWeightsSha256']
        elif initial_digest != info['initialWeightsSha256']:
            raise RuntimeError('Completed arms have different initial weights.')
        arm_results[arm] = info
        print('Verified completed arm:', arm)
        continue
    arm_dir.mkdir(exist_ok=False)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model, base_model = make_model()
    digest = weight_digest(model)
    if initial_digest is None:
        initial_digest = digest
    elif initial_digest != digest:
        raise RuntimeError('Arms did not start from identical weights.')
    augmentation = make_augmentation()
    histories = []
    completed_epochs = 0
    for stage, epochs, learning_rate in ((1, STAGE1_EPOCHS, STAGE1_LR),
                                          (2, STAGE2_EPOCHS, STAGE2_LR)):
        if stage == 2:
            base_model.trainable = True
            for layer in base_model.layers:
                if isinstance(layer, tf.keras.layers.BatchNormalization):
                    # Freeze pretrained running statistics with this small dataset.
                    layer.trainable = False
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        checkpoint = arm_dir / f'stage{stage}_best.weights.h5'
        best_loss = float('inf')
        for stage_epoch in range(epochs):
            print(arm, 'stage', stage, 'epoch', stage_epoch+1, '/', epochs)
            history = model.fit(make_dataset(train_rows, arm, completed_epochs, augmentation),
                validation_data=make_dataset(val_rows), epochs=1,
                class_weight=class_weight, verbose=2)
            metrics = {key: float(values[-1]) for key, values in history.history.items()}
            if not all(math.isfinite(value) for value in metrics.values()):
                raise RuntimeError('Non-finite training metrics; arm is incomplete.')
            histories.append({'stage': stage, 'epoch': completed_epochs+1, **metrics})
            # Select minimum validation loss separately within each stage, exactly as V4.1.1.
            if metrics['val_loss'] < best_loss:
                best_loss = metrics['val_loss']
                model.save_weights(checkpoint)
            completed_epochs += 1
        model.load_weights(checkpoint)
    verify_records_unchanged(records)
    for scene in background_records:
        if sha256_file(scene['path']) != scene['sha256']:
            raise RuntimeError('Background scene changed during training: ' + scene['relativePath'])
    write_json_once(arm_dir/'training_history.json', histories)
    temperature, calibration, probabilities = calibrate_arm(model, val_rows)
    model_path = arm_dir/'model_calibrated.keras'
    model.save(model_path)
    reloaded = tf.keras.models.load_model(model_path, compile=False, safe_mode=True)
    np.testing.assert_allclose(probabilities, predict_records(reloaded, val_rows), rtol=1e-4, atol=1e-5)
    probability_path = arm_dir/'validation_probabilities.npy'
    np.save(probability_path, probabilities)
    info = {'arm': arm, 'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES,
        'seed': SEED, 'temperature': temperature, 'temperatureFoldedIntoWeights': True,
        'confidenceThreshold': CONFIDENCE_THRESHOLD,
        'minimumValidationRetention': MIN_ID_RETENTION,
        'preprocessing': PREPROCESSING, 'classWeights':
            {CLASS_NAMES[i]: class_weight[i] for i in class_weight},
        'validationCalibrationNll': calibration, 'validationImageCount': len(val_rows),
        'validationProbabilityOrder': 'input_manifest.json images subset=val in listed order',
        'modelSha256': sha256_file(model_path),
        'validationProbabilitiesSha256': sha256_file(probability_path),
        'initialWeightsSha256': initial_digest, 'inputManifestSha256': MANIFEST_SHA256,
        'datasetSplitSha256': DATASET_SPLIT_SHA256, 'planSha256': PLAN_SHA256,
        'whiteReviewSha256': WHITE_REVIEW_SHA256, 'maskReviewSha256': MASK_REVIEW_SHA256,
        'lightingReviewSha256': LIGHT_REVIEW_SHA256,
        'backgroundScheduleSha256': SCHEDULE_SHA256, 'sceneReviewSha256': SCENE_REVIEW_SHA256,
        'recoveryStateSha256': RECOVERY_STATE_SHA256,
        'trainingHistorySha256': sha256_file(arm_dir/'training_history.json'), 'runtime': RUNTIME,
        'baseLayerName': base_model.name, 'testPredictionsRun': False,
        'status': 'experimental candidate; rejection thresholds not fitted'}
    write_json_once(arm_dir/'resume_info.json', info)
    arm_results[arm] = info
    print('Saved calibrated arm:', arm, 'temperature:', temperature)
print('One calibrated seven-class combined model saved. TensorFlow.js export follows.')

### Step 20: Recover export after a Colab disconnect

- **Purpose:** Optionally restore only a fully saved calibrated arm, without fitting weights again.
- **Reads:** Exact saved V4.3.0 run, recovery ledger, plan, reviews, model, history, and validation probabilities.
- **Saves:** Restored export variables only.
- **Check:** In a fresh runtime run Steps 1–2, this step, then 21–24. Source/background paths and notes may remain empty during export recovery; only the saved V4.3.0 run is read. Hash changes or incomplete arms block recovery.

In [ ]:
# Optional export-only recovery: after a disconnect, run Steps 1–2, then this step,
# then Steps 21–24. Skip source preflight/reviews and model.fit.
EXPORT_ONLY_RUN_DIR = ''  # Full completed-arm V4.3.0 training path.
if EXPORT_ONLY_RUN_DIR.strip():
    RUN_DIR = Path(EXPORT_ONLY_RUN_DIR.strip()).resolve()
    if not RUN_DIR.is_relative_to(OUTPUT_ROOT.resolve()) or RUN_DIR == OUTPUT_ROOT.resolve():
        raise ValueError('Recovery run must stay inside the V4.3.0 training output folder.')
    if (RUN_DIR/'training_complete.json').exists():
        raise RuntimeError('This run is already complete; go to Step 25 for diagnostics.')
    plan = read_json(RUN_DIR/'plan.json')
    recovery_state = read_json(RUN_DIR/'recovery_state.json')
    PLAN_SHA256 = sha256_file(RUN_DIR/'plan.json')
    RECOVERY_STATE_SHA256 = sha256_file(RUN_DIR/'recovery_state.json')
    if (recovery_state['modelVersion'] != MODEL_VERSION or recovery_state['classes'] != CLASS_NAMES or
        recovery_state['armOrder'] != list(ARMS) or recovery_state['planSha256'] != PLAN_SHA256):
        raise RuntimeError('Saved recovery ledger, class order, or plan disagrees.')
    verify_artifacts(RUN_DIR, recovery_state['artifacts'])
    expected_settings = {'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES, 'arms': list(ARMS),
        'seed': SEED, 'batchSize': BATCH_SIZE, 'imageSize': list(IMAGE_SIZE),
        'stage1': [STAGE1_EPOCHS, STAGE1_LR], 'stage2': [STAGE2_EPOCHS, STAGE2_LR],
        'augmentation': [AUG_FLIP, AUG_ROTATION_DEG, AUG_ZOOM, AUG_SHEAR, AUG_BRIGHTNESS],
        'blendProbability': BLEND_PROBABILITY, 'localLightProbability': LOCAL_LIGHT_PROBABILITY,
        'lightAmplitude': list(LIGHT_AMPLITUDE), 'lightRadius': list(LIGHT_RADIUS), 'preprocessing': PREPROCESSING}
    if any(plan.get(k) != value for k, value in expected_settings.items()):
        raise RuntimeError('Saved recovery settings disagree with the unchanged V4.3.0 recipe.')
    MANIFEST_SHA256 = sha256_file(RUN_DIR/'input_manifest.json')
    DATASET_SPLIT_SHA256 = sha256_file(RUN_DIR/'dataset_split.zip')
    if plan['inputManifestSha256'] != MANIFEST_SHA256 or plan['datasetSplitSha256'] != DATASET_SPLIT_SHA256:
        raise RuntimeError('Saved manifest/split disagrees with plan.')
    WHITE_REVIEW_SHA256 = plan['whiteReviewSha256']
    MASK_REVIEW_SHA256 = plan['maskReviewSha256']
    LIGHT_REVIEW_SHA256 = plan['lightingReviewSha256']
    SCENE_REVIEW_SHA256 = plan['sceneReviewSha256']
    SCENE_INVENTORY_SHA256 = plan['sceneInventorySha256']
    SCHEDULE_SHA256 = plan['backgroundScheduleSha256']
    ARM_ROOT = RUN_DIR/'arms'
    arm_results, initial_digest = {}, None
    for arm in ARMS:
        folder = ARM_ROOT/arm
        info = read_json(folder/'resume_info.json')
        contract = {'arm': arm, 'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES,
            'planSha256': PLAN_SHA256, 'inputManifestSha256': MANIFEST_SHA256,
            'datasetSplitSha256': DATASET_SPLIT_SHA256, 'whiteReviewSha256': WHITE_REVIEW_SHA256,
            'maskReviewSha256': MASK_REVIEW_SHA256, 'lightingReviewSha256': LIGHT_REVIEW_SHA256,
            'sceneReviewSha256': SCENE_REVIEW_SHA256, 'backgroundScheduleSha256': SCHEDULE_SHA256,
            'recoveryStateSha256': RECOVERY_STATE_SHA256, 'temperatureFoldedIntoWeights': True,
            'testPredictionsRun': False, 'validationImageCount': 391}
        if any(info.get(k) != value for k, value in contract.items()):
            raise RuntimeError('Saved arm metadata disagrees with reviewed inputs.')
        verify_artifacts(folder, {'model_calibrated.keras': info['modelSha256'],
            'validation_probabilities.npy': info['validationProbabilitiesSha256'],
            'training_history.json': info['trainingHistorySha256']})
        values = np.load(folder/'validation_probabilities.npy', allow_pickle=False)
        if (values.shape != (391, 7) or not np.isfinite(values).all() or
            np.any(values < 0) or not np.allclose(values.sum(axis=1), 1, atol=1e-5)):
            raise ValueError('Saved validation probabilities are not 391 seven-class probability rows.')
        initial_digest = info['initialWeightsSha256']
        arm_results[arm] = info
    print('Ready for export-only Steps 21–24:', RUN_DIR)
else:
    print('No export-only recovery requested; keep the current verified training state.')

### Step 21: Draw actual saved training curves

- **Purpose:** Plot the real 15-epoch losses/accuracies and the boundary between frozen-head training and fine-tuning.
- **Reads:** Hash-checked training_history.json from combined.
- **Saves:** figures/training_curves.png.
- **Check:** One combined row with two charts; history must be 5 + 10 epochs. No model.fit or new prediction occurs.

In [ ]:
# Plot saved epoch metrics only; no training or inference occurs in this step.
import matplotlib.pyplot as plt
figure_dir = RUN_DIR/'figures'
figure_dir.mkdir(exist_ok=True)
figure_path = figure_dir/'training_curves.png'
curve_record_path = RUN_DIR/'training_curves_provenance.json'
history_path = ARM_ROOT/'combined'/'training_history.json'
expected_history = arm_results['combined']['trainingHistorySha256']
if sha256_file(history_path) != expected_history:
    raise RuntimeError('Saved epoch history changed before curve plotting.')
if figure_path.exists() or curve_record_path.exists():
    # A converter failure may have happened after the curve was already saved.
    # Reuse the checked curve on export recovery, without overwriting its evidence.
    if not figure_path.is_file() or not curve_record_path.is_file():
        raise RuntimeError('Incomplete saved training curve/provenance pair; inspect the run before export.')
    saved_curve = read_json(curve_record_path)
    if (saved_curve['historySha256'] != expected_history or saved_curve['planSha256'] != PLAN_SHA256 or
        saved_curve['figureSha256'] != sha256_file(figure_path)):
        raise RuntimeError('Saved curve or its provenance changed.')
    print('Verified existing training curve:', figure_path)
else:
    history = read_json(history_path)
    if (len(history) != 15 or [r['epoch'] for r in history] != list(range(1, 16)) or
        [r['stage'] for r in history] != [1]*5 + [2]*10):
        raise RuntimeError('Incomplete or unexpected combined training history.')
    epochs = [r['epoch'] for r in history]
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), squeeze=False)
    for column, (metric, label) in enumerate((('loss', 'Cross-entropy loss'), ('accuracy', 'Accuracy'))):
        if any(metric not in r or 'val_'+metric not in r or
               not math.isfinite(r[metric]) or not math.isfinite(r['val_'+metric]) for r in history):
            raise RuntimeError('Missing or non-finite combined epoch metric.')
        axis = axes[0, column]
        axis.plot(epochs, [r[metric] for r in history], label='Train')
        axis.plot(epochs, [r['val_'+metric] for r in history], label='Validation')
        axis.axvline(5.5, color='gray', linestyle=':', label='Fine-tuning starts')
        axis.set(title='Combined - '+label, ylabel=label, xlabel='Epoch')
        axis.grid(alpha=0.25)
        axis.legend()
    figure.suptitle('V4.3.0 seven-class combined: actual saved training metrics')
    figure.tight_layout()
    figure.savefig(figure_path, dpi=160)
    plt.close(figure)
    write_json_once(curve_record_path, {'planSha256': PLAN_SHA256, 'historySha256': expected_history,
        'figureSha256': sha256_file(figure_path), 'source': 'actual saved epoch metrics; no model.fit'})
    print('Saved actual-history figure:', figure_path)

### Step 22: Define TensorFlow.js export checks

- **Purpose:** Validate graph input/output dimensions, shard boundaries, finite values, and float32 export.
- **Reads:** TensorFlow.js graph/weight files generated in the next steps.
- **Saves:** Export validation helper only.
- **Check:** Exactly seven outputs and 224 x 224 RGB input; quantized weights are rejected to retain the V4.1.1 export recipe.

In [ ]:
def verify_tfjs_export(folder):
    folder = Path(folder)
    graph = json.loads((folder / 'model.json').read_text(encoding='utf-8'))
    if graph.get('format') != 'graph-model':
        raise ValueError('Expected a TensorFlow.js Graph Model.')
    signature = graph.get('signature', {})
    outputs = list(signature.get('outputs', {}).values())
    inputs = list(signature.get('inputs', {}).values())
    dims = lambda tensor: [int(d['size']) for d in tensor['tensorShape']['dim']]
    if len(outputs) != 1 or dims(outputs[0])[-1:] != [len(CLASS_NAMES)]:
        raise ValueError('Export must expose seven output probabilities.')
    if len(inputs) != 1 or dims(inputs[0])[1:] != [224, 224, 3]:
        raise ValueError('Export input shape differs from the training plan.')
    dtypes = {'float32': '<f4', 'int32': '<i4', 'bool': '?'}
    used = set()
    for group in graph['weightsManifest']:
        parts = []
        for name in group['paths']:
            if name in used:
                raise ValueError('Repeated weight shard: ' + name)
            used.add(name)
            parts.append(inside(folder, name).read_bytes())
        buffer, offset = b''.join(parts), 0
        for weight in group['weights']:
            if 'quantization' in weight or weight['dtype'] not in dtypes:
                raise ValueError('This experiment requires unquantized float32 weights.')
            dtype = np.dtype(dtypes[weight['dtype']])
            count = math.prod(weight['shape'])
            size = count * dtype.itemsize
            if offset + size > len(buffer):
                raise ValueError('Truncated weight shard.')
            values = np.frombuffer(buffer[offset:offset + size], dtype=dtype)
            if not np.isfinite(values).all():
                raise ValueError('Non-finite values found in the exported weights.')
            offset += size
        if offset != len(buffer):
            raise ValueError('Weight manifest and shard sizes disagree.')
    if not used:
        raise ValueError('No exported weight shards.')
    return ['model.json'] + sorted(used)

### Step 23: Define the converter with the existing compatibility fallbacks

- **Purpose:** Export the calibrated Keras model through a SavedModel and retain the V4.1.1 converter recovery strategy.
- **Reads:** Saved calibrated model and Colab conversion libraries.
- **Saves:** Helper only until called; converter attempts get unique local output folders.
- **Check:** TensorFlow/NumPy are preserved; unused converter imports may be upgraded or isolated in subprocesses. Failures do not create training_complete.json.

In [ ]:
def convert_one_arm(arm_dir):
    EXPORT_DIR = arm_dir / 'export_attempts' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])
    EXPORT_DIR.mkdir(parents=True, exist_ok=False)
    SAVED_MODEL = str(EXPORT_DIR / 'saved_model')
    TFJS_DIR = str(EXPORT_DIR / 'model')
    model = tf.keras.models.load_model(arm_dir / 'model_calibrated.keras', compile=False, safe_mode=True)
    model.export(SAVED_MODEL, verbose=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'tensorflowjs==4.22.0'], check=True)
    import re, os, glob
    import importlib, importlib.util
    import importlib.metadata as importlib_metadata

    # Stubs live here, not in dist-packages, and are handed to the converter through
    # PYTHONPATH. Real installed packages are left untouched.
    STUB_DIR = str(EXPORT_DIR / "converter_stubs")

    # Module name as imported -> name pip knows it by. None means no wheel exists for
    # this Python version, so the module gets stubbed instead of installed.
    PIP_NAME = {
        "tensorflow_decision_forests": None,
        "tensorflow_hub": "tensorflow-hub",
        "tf_keras": "tf-keras==" + ".".join(tf.__version__.split(".")[:2]) + ".0",
        "pkg_resources": "setuptools<81",
        "jax": "jax",
        "jaxlib": "jaxlib",
        "flax": "flax",
        "packaging": "packaging",
    }

    # Never stub these. If the failure is really inside TensorFlow itself, stubbing it
    # would hide the problem rather than fix it.
    NEVER_STUB = ("tensorflow", "numpy", "tensorflowjs")


    def tfjs_dirs():
        # find_spec locates the package without running its __init__, so this works
        # even when importing tensorflowjs would raise.
        importlib.invalidate_caches()
        spec = importlib.util.find_spec("tensorflowjs")
        if spec is None or not spec.submodule_search_locations:
            return None
        return list(spec.submodule_search_locations)[0]


    def version_of(dist):
        importlib.invalidate_caches()
        try:
            return importlib_metadata.version(dist)
        except Exception:
            return "not installed"


    def patch_numpy_aliases(pkg_dir):
        # numpy 1.24 removed np.object, np.bool, np.int, np.float and np.str. Older
        # tensorflowjs builds reference them at import time. The \b boundary keeps the
        # rewrite off np.object_, np.bool_, np.float32 and friends. Re-run after every
        # pip install, in case pip moved to an older build.
        if not pkg_dir:
            return []
        names = ("object", "bool", "int", "float", "str", "complex")
        touched = []
        for path in glob.glob(os.path.join(pkg_dir, "**", "*.py"), recursive=True):
            try:
                with open(path, encoding="utf-8") as fh:
                    src = fh.read()
            except (OSError, UnicodeDecodeError):
                continue
            new = src
            for n in names:
                new = re.sub(r"\bnp\.%s\b" % n, n, new)
                new = re.sub(r"\bnumpy\.%s\b" % n, n, new)
            if new != src:
                with open(path, "w", encoding="utf-8") as fh:
                    fh.write(new)
                touched.append(os.path.relpath(path, pkg_dir))
        return touched


    def write_stub(module_name):
        # Permissive placeholder. tensorflowjs imports these at load time but only uses
        # them for input formats this project never produces, so an empty shell is
        # enough to get past the import.
        d = os.path.join(STUB_DIR, module_name)
        os.makedirs(d, exist_ok=True)
        body = (
            "__version__ = '0.0.0-stub'\n"
            "\n"
            "class _Any(object):\n"
            "    def __getattr__(self, k):\n"
            "        return _Any\n"
            "    def __call__(self, *a, **k):\n"
            "        return _Any()\n"
            "\n"
            "def __getattr__(name):\n"
            "    return _Any\n"
        )
        for fname in ("__init__.py", "keras.py"):
            with open(os.path.join(d, fname), "w") as fh:
                fh.write(body)


    def pip_install(pip_name, upgrade=True):
        # --no-deps is deliberate. A plain install can change the TensorFlow version,
        # which forces a runtime restart and loses the trained model held in memory.
        args = [sys.executable, "-m", "pip", "install", "-q", "--no-deps"]
        if upgrade:
            args.append("-U")
        args.append(pip_name)
        return subprocess.run(args, capture_output=True, text=True).returncode == 0


    def run_converter():
        # List form, no shell. A shell would glob-expand the * in --quantize_float16=*.
        quant = ["--quantize_float16=*"] if QUANTIZE_FLOAT16 else []
        cmd = ([sys.executable, "-c", "from importlib.metadata import distribution; next(e for e in distribution('tensorflowjs').entry_points if e.name == 'tensorflowjs_converter').load()()", "--input_format=tf_saved_model"]
               + quant + [SAVED_MODEL, TFJS_DIR])
        env = dict(os.environ)
        env["TF_CPP_MIN_LOG_LEVEL"] = "2"
        if os.path.isdir(STUB_DIR):
            env["PYTHONPATH"] = STUB_DIR + os.pathsep + env.get("PYTHONPATH", "")
        print("     $", " ".join(cmd))
        return subprocess.run(cmd, capture_output=True, text=True, env=env)


    def blame(text):
        """Which third-party package is failing to import inside the converter."""
        text = text or ""
        m = re.search(r"No module named '([^']+)'", text)
        if m:
            return m.group(1).split(".")[0]
        # Otherwise take the deepest traceback frame that sits in site-packages and
        # belongs to neither tensorflowjs nor tensorflow.
        found = None
        for path in re.findall(r'File "([^"]+)", line \d+, in <module>', text):
            parts = path.replace("\\", "/").split("/")
            for marker in ("dist-packages", "site-packages"):
                if marker in parts:
                    top = parts[parts.index(marker) + 1]
                    if top.endswith(".py"):
                        top = top[:-3]
                    if top not in NEVER_STUB:
                        found = top
        return found


    def tail(text, n=25):
        lines = [l for l in (text or "").strip().split("\n") if l.strip()]
        return "\n".join(lines[-n:])


    os.makedirs(STUB_DIR, exist_ok=True)
    pkg = tfjs_dirs()
    print("numpy           :", np.__version__)
    print("tensorflow      :", tf.__version__)
    print("tensorflowjs    :", version_of("tensorflowjs"))
    print("tensorflow_hub  :", version_of("tensorflow-hub"))
    print("package dir     :", pkg)
    print("")
    if pkg is None:
        raise RuntimeError("tensorflowjs is not installed. Install converter dependencies in this cell, then retry this export step.")

    # --- Step A: tensorflow_hub 0.15 and older import tf.compat.v1.estimator, -----
    # --- which TensorFlow removed in 2.16. Move to a release that does not. --------
    print("Step A - upgrading tensorflow_hub (old builds need the removed Estimator API)")
    pip_install("tensorflow-hub")
    print("        tensorflow_hub :", version_of("tensorflow-hub"))
    patch_numpy_aliases(tfjs_dirs())
    proc = run_converter()

    # --- Step B: repair loop. Upgrade the offending package once, then stub it. ----
    tried = {}
    for _ in range(8):
        if proc.returncode == 0:
            break
        mod = blame((proc.stderr or "") + "\n" + (proc.stdout or ""))
        if mod is None or mod in NEVER_STUB or mod not in PIP_NAME:
            break
        pip_name = PIP_NAME.get(mod, mod)
        stage = tried.get(mod, 0)
        print("")
        if stage == 0 and pip_name:
            print("Step B - {} is breaking the import, upgrading it".format(mod))
            if not pip_install(pip_name):
                print("        pip could not install {}".format(pip_name))
            else:
                print("        {} :".format(mod), version_of(pip_name))
            tried[mod] = 1
        elif stage <= 1 and mod in {"tensorflow_decision_forests", "tensorflow_hub", "jax", "jaxlib", "flax"}:
            print("Step B - {} still unusable, shadowing it with a stub".format(mod))
            print("        the SavedModel path does not use it")
            write_stub(mod)
            tried[mod] = 2
        else:
            break
        patch_numpy_aliases(tfjs_dirs())
        proc = run_converter()


    model_json = os.path.join(TFJS_DIR, 'model.json')
    if proc.returncode != 0 or not os.path.isfile(model_json):
        print(tail(proc.stderr) or tail(proc.stdout))
        raise RuntimeError('TensorFlow.js converter failed; inspect this arm attempt, then retry export.')
    return Path(TFJS_DIR)

### Step 24: Export and close the training run

- **Purpose:** Export the one calibrated seven-class model and freeze all shared/model/browser artifact hashes.
- **Reads:** Verified inputs/reviews, saved model/history, actual curves, and completed graph/weight files.
- **Saves:** arms/combined/tfjs_model/, requirements.txt, and training_complete.json.
- **Check:** Only a fully hash-verified export is complete. testPredictionsRun remains false. Browser config has no fitted margin/entropy gates; use the separate threshold notebook.

In [ ]:
# Keep the calibrated export unquantized, matching the preserved V4.1.1 recipe.
QUANTIZE_FLOAT16 = False
if (RUN_DIR/'training_complete.json').exists():
    raise RuntimeError('training_complete.json already exists. Use the saved run in the threshold notebook.')
for arm in ARMS:
    arm_dir = ARM_ROOT / arm
    info = arm_results[arm]
    if sha256_file(arm_dir/'model_calibrated.keras') != info['modelSha256']:
        raise RuntimeError('Calibrated model changed before export: ' + arm)
    target = arm_dir / 'tfjs_model'
    if not target.exists():
        attempt = convert_one_arm(arm_dir)
        names = verify_tfjs_export(attempt)
        config = {'modelVersion': MODEL_VERSION, 'arm': arm, 'classes': CLASS_NAMES,
            'inputSize': list(IMAGE_SIZE), 'preprocessing': PREPROCESSING,
            'temperature': info['temperature'], 'temperatureFoldedIntoWeights': True,
            'status': 'experimental candidate; rejection thresholds not fitted',
            'thresholds': {'confidenceThreshold': CONFIDENCE_THRESHOLD,
                'marginThreshold': None, 'entropyThreshold': None}}
        write_json_once(attempt/'model_config.json', config)
        shutil.move(str(attempt), str(target))
    names = verify_tfjs_export(target) + ['model_config.json']
    config = json.loads((target/'model_config.json').read_text(encoding='utf-8'))
    if (config['classes'] != CLASS_NAMES or config['arm'] != arm or config['modelVersion'] != MODEL_VERSION or
        config.get('temperatureFoldedIntoWeights') is not True or config['temperature'] != info['temperature'] or
        config['preprocessing'] != PREPROCESSING):
        raise RuntimeError('TensorFlow.js model config disagrees with this arm.')
    info['tfjsDirectory'] = (target.relative_to(RUN_DIR)).as_posix()
    info['tfjsArtifacts'] = {name: sha256_file(target/name) for name in names}
    print('Verified TensorFlow.js export:', arm, len(names), 'files')

recovery_state = read_json(RUN_DIR/'recovery_state.json')
verify_artifacts(RUN_DIR, recovery_state['artifacts'])
# Save runtime dependencies before declaring completion, so failure cannot leave
# a completed run without its promised reproduction record.
requirements_path = RUN_DIR/'requirements.txt'
if requirements_path.exists():
    if not requirements_path.read_text(encoding='utf-8').strip():
        raise RuntimeError('Existing runtime dependency record is empty; investigate the incomplete export attempt.')
    print('Keeping the runtime record saved during an earlier export attempt:', requirements_path)
else:
    requirements_text = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    if not requirements_text.strip():
        raise RuntimeError('Runtime dependency capture was empty; completion is blocked.')
    requirements_temporary = requirements_path.with_name('requirements.txt.partial-' + uuid.uuid4().hex[:8])
    with requirements_temporary.open('x', encoding='utf-8') as handle:
        handle.write(requirements_text)
    requirements_temporary.rename(requirements_path)
artifacts = {**recovery_state['artifacts'],
    'recovery_state.json': RECOVERY_STATE_SHA256,
    'requirements.txt': sha256_file(requirements_path),
    'training_curves_provenance.json': sha256_file(RUN_DIR/'training_curves_provenance.json'),
    'figures/training_curves.png': sha256_file(RUN_DIR/'figures/training_curves.png')}
for arm in ARMS:
    arm_dir = ARM_ROOT / arm
    for name in ('model_calibrated.keras', 'validation_probabilities.npy',
                 'resume_info.json', 'training_history.json'):
        relative = f'arms/{arm}/{name}'
        artifacts[relative] = sha256_file(RUN_DIR/relative)
    for name, digest in arm_results[arm]['tfjsArtifacts'].items():
        artifacts[f'arms/{arm}/tfjs_model/{name}'] = digest
verify_artifacts(RUN_DIR, artifacts)
write_json_once(RUN_DIR/'training_complete.json', {
    'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES, 'armOrder': list(ARMS),
    'arms': arm_results, 'initialWeightsSha256': initial_digest,
    'inputManifestSha256': MANIFEST_SHA256, 'datasetSplitSha256': DATASET_SPLIT_SHA256,
    'planSha256': PLAN_SHA256, 'whiteReviewSha256': WHITE_REVIEW_SHA256,
    'maskReviewSha256': MASK_REVIEW_SHA256, 'lightingReviewSha256': LIGHT_REVIEW_SHA256,
    'backgroundScheduleSha256': SCHEDULE_SHA256, 'sceneReviewSha256': SCENE_REVIEW_SHA256,
    'sceneInventorySha256': SCENE_INVENTORY_SHA256, 'recoveryStateSha256': RECOVERY_STATE_SHA256,
    'artifacts': artifacts, 'testPredictionsRun': False,
    'finishedAt': datetime.now(timezone.utc).isoformat(),
    'status': 'experimental candidate; evaluate before adoption'})
print('TRAINING COMPLETE: one seven-class combined experimental model, thresholds not fitted. Paste this exact path into MangoLeaf_AI_Threshold_Fitting_v4_3_0.ipynb:')
print(RUN_DIR)

## Training diagnostics and the return from threshold fitting

Run Steps 25–26 after Step 24; these reconstruct one representative train augmentation and draw Grad-CAM on seven validation photos. They reuse weights and never call `fit`. The final Keras augmentation transforms are new representative random draws because historical per-batch tensors were not saved. Actual curves already came from the saved fit history.

Then run `MangoLeaf_AI_Threshold_Fitting_v4_3_0.ipynb` against this exact training path. It freezes selected gates before reading test/report predictions. Return here for Steps 27–30. Final confusion, class, and threshold-search figures read its saved reports and perform no new inference. After disconnect, run Step 25 onward with the exact saved paths. Completed evidence is immutable; inspect existing figures instead of overwriting them.

### Step 25: Restore the completed run and define figure helpers

- **Purpose:** Read only the completed training evidence and reconstruct its exact preprocessing and pre-Keras scene/light policy.
- **Reads:** Completion/plan/manifest/ZIP hashes, saved masks, and frozen scene copies from both original source pools.
- **Saves:** Figure helper functions only.
- **Check:** Works alone after disconnect. Full seven-class order, combined only, exact split counts, and every completed artifact hash must agree. No original scene-root lookup is needed.

In [ ]:
# This cell is self-contained for a new runtime after a Colab disconnect.
from google.colab import drive
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import shutil, uuid
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
FIGURE_TRAINING_RUN = ''  # Full saved run path or its exact v4-3-0-training-* folder name.
TRAINING_RUN_ROOT = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.3.0/training')
if FIGURE_TRAINING_RUN.strip():
    supplied = Path(FIGURE_TRAINING_RUN.strip())
    if supplied.is_absolute():
        FIG_TRAINING = supplied.resolve()
    elif supplied.name == str(supplied):
        FIG_TRAINING = (TRAINING_RUN_ROOT/supplied).resolve()
    else:
        raise ValueError('Use the full Drive path or exact run folder name.')
elif 'RUN_DIR' in globals():
    FIG_TRAINING = Path(RUN_DIR).resolve()
else:
    raise ValueError('Set FIGURE_TRAINING_RUN to the exact saved training run.')
if FIG_TRAINING == TRAINING_RUN_ROOT.resolve() or not FIG_TRAINING.is_relative_to(TRAINING_RUN_ROOT.resolve()):
    raise ValueError('Figure source must be one run inside the V4.3.0 training folder.')
if not (FIG_TRAINING/'training_complete.json').is_file():
    raise FileNotFoundError('Finish Step 24 before diagnostics.')
"""Draw V4.3.0 seven-class combined candidate figures from a saved, hash-checked run.

These helpers live in the training notebook. They never train a model or change frozen threshold choices.
"""

import argparse
import csv
import hashlib
import io
import json
import random
import tempfile
import zipfile
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps


ARMS = ("combined",)
SEED = 42
IMAGE_SIZE = (224, 224)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    with Path(path).open("rb") as handle:
        digest = hashlib.sha256()
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def checked_zip_image(bundle, row):
    name = row["relativePath"]
    if name.startswith("/") or ".." in Path(name).parts or "\\" in name:
        raise ValueError("Unsafe image path in manifest: " + name)
    data = bundle.read(name)
    if len(data) != row["bytes"] or sha256_bytes(data) != row["sha256"]:
        raise ValueError("Dataset image differs from manifest: " + name)
    return data


def figure_load_pixels(data, suffix):
    # Match training's RGB decode and 224x224 bilinear resize, not Pillow resize.
    if suffix.lower() == ".webp":
        with Image.open(io.BytesIO(data)) as picture:
            pixels = np.asarray(ImageOps.exif_transpose(picture).convert("RGB"), dtype=np.float32)
    else:
        pixels = tf.io.decode_image(data, channels=3, expand_animations=False)
        pixels.set_shape((None, None, 3))
    return tf.image.resize(pixels, IMAGE_SIZE, method="bilinear", antialias=False).numpy()


def stable_seed(*parts):
    return int.from_bytes(hashlib.sha256("|".join(map(str, parts)).encode()).digest()[:8], "big")


def light_is_applied(key, epoch, probability):
    return random.Random(stable_seed(SEED, key, epoch, "local-light")).random() < probability


def figure_add_local_light(pixels, key, epoch, plan):
    # Same order of random draws as the V4.3.0 combined training loader.
    rng = random.Random(stable_seed(SEED, key, epoch, "local-light"))
    if rng.random() >= plan["localLightProbability"]:
        return pixels
    sign = rng.choice((-1, 1))
    amplitude = rng.uniform(*plan["lightAmplitude"])
    radius = rng.uniform(*plan["lightRadius"]) * min(IMAGE_SIZE)
    cx, cy = rng.uniform(0, IMAGE_SIZE[1] - 1), rng.uniform(0, IMAGE_SIZE[0] - 1)
    yy, xx = np.mgrid[: IMAGE_SIZE[0], : IMAGE_SIZE[1]]
    field = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * radius**2))[..., None]
    return np.clip(pixels + sign * amplitude * field, 0, 255).astype(np.float32)


class FigureRandomShear(tf.keras.layers.Layer):
    # Match the custom training layer so preview uses the same transform recipe.
    def __init__(self, factor=0.2, fill_mode="NEAREST", **kwargs):
        super().__init__(**kwargs)
        self.factor = factor
        self.fill_mode = fill_mode

    def call(self, inputs, training=False):
        if not training:
            return inputs
        shape = tf.shape(inputs)
        n = shape[0]
        h = tf.cast(shape[1], tf.float32)
        w = tf.cast(shape[2], tf.float32)
        cx, cy = w / 2.0, h / 2.0
        sx = tf.random.uniform([n], -self.factor, self.factor)
        sy = tf.random.uniform([n], -self.factor, self.factor)
        one = tf.ones_like(sx)
        zero = tf.zeros_like(sx)
        transforms = tf.stack([one, sx, -sx * cy, sy, one, -sy * cx, zero, zero], axis=1)
        return tf.raw_ops.ImageProjectiveTransformV3(
            images=inputs, transforms=transforms, output_shape=shape[1:3],
            fill_value=0.0, interpolation="BILINEAR", fill_mode=self.fill_mode)


def figure_make_augmentation(plan):
    flip, rotation, zoom, shear, brightness = plan["augmentation"]
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip(flip, seed=SEED),
        tf.keras.layers.RandomRotation(rotation / 360.0, fill_mode="nearest", seed=SEED),
        tf.keras.layers.RandomZoom(zoom, fill_mode="nearest", seed=SEED),
        FigureRandomShear(shear),
        tf.keras.layers.RandomBrightness(brightness, value_range=(0.0, 255.0), seed=SEED),
    ])


def save_augmentation_preview(training, output, plan, manifest, bundle):
    schedule = read_json(training / "background_schedule.json")["presentations"]
    mask_rows = {r["relativePath"]: r for r in read_json(training / "mask_review.json")["masks"] if r["useForBlend"]}
    rows = {r["relativePath"]: r for r in manifest["images"]}
    scene_rows = {r["relativePath"]: r for r in manifest["backgroundImages"]}
    # Choose a scheduled train image for which both scene and local light apply.
    # A frozen choice is reconstructed; final Keras random draw below is new.
    picked = next((r for r in schedule if r["trainRelativePath"] in mask_rows and
                   light_is_applied(r["trainRelativePath"], r["epoch"], plan["localLightProbability"])), None)
    if picked is None:
        raise ValueError("No approved scheduled scene also received local light")
    key, epoch = picked["trainRelativePath"], picked["epoch"]
    image_row, mask_row, scene_row = rows[key], mask_rows[key], scene_rows[picked["sceneRelativePath"]]
    scene_path = inside(training, scene_row["frozenRelativePath"])
    mask_path = training / "masks" / (mask_row["id"] + ".png")
    if (sha256_file(scene_path) != picked["sceneSha256"] or
            sha256_file(scene_path) != scene_row["sha256"] or
            sha256_file(mask_path) != mask_row["maskSha256"] or
            image_row["sha256"] != mask_row["sha256"]):
        raise ValueError("Preview scene, image, or mask hash differs from frozen run")
    # Also verify that the saved schedule agrees with the original random choice.
    rng = random.Random(stable_seed(SEED, key, epoch, "background"))
    if rng.random() >= plan["blendProbability"] or rng.choice(manifest["backgroundImages"])["sha256"] != scene_row["sha256"]:
        raise ValueError("Saved scene does not match V4.3.0 selection recipe")
    original = figure_load_pixels(checked_zip_image(bundle, image_row), Path(key).suffix)
    with Image.open(mask_path) as picture:
        alpha = np.asarray(picture.convert("L").resize(IMAGE_SIZE, Image.Resampling.BILINEAR), dtype=np.float32)[..., None] / 255
    with Image.open(scene_path) as picture:
        background = np.asarray(ImageOps.fit(picture.convert("RGB"), IMAGE_SIZE, method=Image.Resampling.BILINEAR), dtype=np.float32)
    replaced = original * alpha + background * (1 - alpha)
    pre = [original, replaced, figure_add_local_light(original, key, epoch, plan), figure_add_local_light(replaced, key, epoch, plan)]
    # New stochastic draws demonstrate the recipe; original per-epoch draws were not saved.
    tf.keras.utils.set_random_seed(SEED)
    samples = [figure_make_augmentation(plan)(tf.convert_to_tensor(x[None], dtype=tf.float32), training=True).numpy()[0] for x in pre]
    fig, axes = plt.subplots(2, 4, figsize=(14, 8), constrained_layout=True)
    for column, arm in enumerate(("Original", "Scene demonstration", "Light demonstration", "Combined input")):
        for row_index, pixels in enumerate((pre[column], samples[column])):
            axes[row_index, column].imshow(np.uint8(np.clip(pixels, 0, 255)))
            axes[row_index, column].axis("off")
            axes[row_index, column].set_title(f"{arm} — {'before Keras aug' if row_index == 0 else 'new Keras aug draw'}")
    fig.suptitle("V4.3.0 one-model augmentation demonstration: combined input reconstructed; final transforms are new")
    fig.savefig(output / "augmentation_preview.png", dpi=160)
    plt.close(fig)
    return {"trainingImage": key, "trainingImageSha256": image_row["sha256"], "epoch": epoch,
            "scene": scene_row["relativePath"], "sceneSha256": scene_row["sha256"],
            "mask": mask_path.name, "maskSha256": mask_row["maskSha256"],
            "interpretation": "Only the Combined input top column reconstructs the scheduled pre-Keras training input; other top columns isolate effects for demonstration. Bottom transforms are new representative draws, not recovered fit tensors."}


def save_grad_cam(training, output, classes, manifest, bundle, expected_model_hash):
    model_path = training / "arms" / "combined" / "model_calibrated.keras"
    if sha256_file(model_path) != expected_model_hash:
        raise ValueError("Calibrated combined model differs from frozen record")
    model = tf.keras.models.load_model(model_path, compile=False, safe_mode=True)
    if model.input_shape != (None, 224, 224, 3) or model.output_shape != (None, len(classes)):
        raise ValueError("Unexpected V4.3.0 seven-class combined model shape")
    base = model.get_layer(FIG_COMPLETE["arms"]["combined"]["baseLayerName"])
    picked = [next(r for r in manifest["images"] if r["subset"] == "val" and r["label"] == label) for label in classes]
    fig, axes = plt.subplots(2, len(classes), figsize=(3 * len(classes), 6), constrained_layout=True)
    rows = []
    for column, row in enumerate(picked):
        pixels = figure_load_pixels(checked_zip_image(bundle, row), Path(row["relativePath"]).suffix)
        batch = tf.convert_to_tensor(pixels[None], dtype=tf.float32)
        with tf.GradientTape() as tape:
            features = base(batch, training=False)
            tape.watch(features)
            hidden = model.get_layer("gap")(features)
            hidden = model.get_layer("dropout")(hidden, training=False)
            logits = model.get_layer("logits")(hidden)
            predicted = int(tf.argmax(logits[0]).numpy())
            score = logits[:, predicted]
        gradients = tape.gradient(score, features)
        if gradients is None:
            raise RuntimeError("Grad-CAM cannot connect the feature map to logits")
        heat = tf.nn.relu(tf.reduce_sum(features[0] * tf.reduce_mean(gradients, axis=(0, 1, 2)), axis=-1))
        heat = heat / (tf.reduce_max(heat) + 1e-9)
        heat = tf.image.resize(heat[..., None], IMAGE_SIZE).numpy()[..., 0]
        probabilities = model(batch, training=False).numpy()[0]
        rows.append({"image": row["relativePath"], "imageSha256": row["sha256"], "actual": row["label"],
                     "rawPrediction": classes[predicted], "rawConfidence": float(probabilities[predicted])})
        for axis in axes[:, column]:
            axis.imshow(np.uint8(np.clip(pixels, 0, 255)))
            axis.axis("off")
        axes[0, column].set_title(row["label"].replace("_", " "), fontsize=9)
        axes[1, column].imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1)
        axes[1, column].set_title(f"{classes[predicted].replace('_', ' ')} {probabilities[predicted]:.2f}", fontsize=9)
    fig.suptitle("V4.3.0 seven-class combined model: fresh Grad-CAM on validation examples; raw scores, no rejection gates")
    fig.savefig(output / "grad_cam.png", dpi=160)
    plt.close(fig)
    return rows



def inside(root, relative):
    relative = PurePosixPath(relative)
    if relative.is_absolute() or '..' in relative.parts or '\\' in str(relative):
        raise ValueError('Unsafe saved artifact path.')
    target = (Path(root)/str(relative)).resolve()
    if target == Path(root).resolve() or not target.is_relative_to(Path(root).resolve()):
        raise ValueError('Saved artifact must stay inside its run.')
    return target


def verify_figure_artifacts(folder, artifacts):
    for relative, expected in artifacts.items():
        path = inside(folder, relative)
        if not path.is_file() or sha256_file(path) != expected:
            raise ValueError('Missing or changed figure source artifact: ' + relative)


def new_figure_attempt(phase):
    # A failed attempt is kept for diagnosis; final files are published together
    # only after every chart and the provenance record have been produced.
    relative = 'figure_attempts/' + phase + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
    attempt = inside(FIG_TRAINING, relative)
    attempt.mkdir(parents=True, exist_ok=False)
    return attempt


def finish_pending_figure_phase(record_name):
    pending_path = inside(FIG_TRAINING, record_name + '.pending.json')
    pending = read_json(pending_path)
    if pending['recordName'] != record_name:
        raise ValueError('Pending figure phase has a different provenance record.')
    attempt = inside(FIG_TRAINING, pending['attemptRelativePath'])
    verify_figure_artifacts(attempt, pending['files'])
    final = FIG_TRAINING/'figures'
    # Check already-published and earlier-phase figures before changing final files.
    for name, expected in pending['record']['figures'].items():
        path = inside(final, name)
        if (name not in pending['files'] or path.exists()) and (not path.is_file() or sha256_file(path) != expected):
            raise ValueError('Existing figure differs from the verified phase: ' + name)
    for name, expected in pending['files'].items():
        target = inside(final, name)
        if target.exists():
            if sha256_file(target) != expected:
                raise ValueError('Existing final figure differs from the verified pending phase: ' + name)
            continue
        # A unique partial file cannot be mistaken for completed output.
        temporary = target.with_name(target.name + '.partial-' + uuid.uuid4().hex[:8])
        shutil.copyfile(inside(attempt, name), temporary)
        if sha256_file(temporary) != expected:
            raise ValueError('Figure copy differs from the staged source: ' + name)
        temporary.rename(target)
    provenance_path = inside(FIG_TRAINING, record_name)
    if provenance_path.exists():
        if read_json(provenance_path) != pending['record']:
            raise ValueError('Existing figure provenance differs from the pending phase.')
    else:
        with provenance_path.open('x', encoding='utf-8') as handle:
            json.dump(pending['record'], handle, indent=2, allow_nan=False)
            handle.write('\n')
    verify_figure_artifacts(final, pending['record']['figures'])
    return pending['record']


def publish_figure_phase(attempt, record_name, record, names):
    files = {name: sha256_file(inside(attempt, name)) for name in names}
    pending_path = inside(FIG_TRAINING, record_name + '.pending.json')
    if pending_path.exists():
        raise FileExistsError('A verified figure phase is pending; resume it before making another attempt.')
    with pending_path.open('x', encoding='utf-8') as handle:
        json.dump({'recordName': record_name, 'attemptRelativePath': attempt.resolve().relative_to(FIG_TRAINING.resolve()).as_posix(),
                   'files': files, 'record': record}, handle, indent=2, allow_nan=False)
        handle.write('\n')
    return finish_pending_figure_phase(record_name)


FIG_PLAN = read_json(FIG_TRAINING/'plan.json')
FIG_COMPLETE = read_json(FIG_TRAINING/'training_complete.json')
FIG_MANIFEST = read_json(FIG_TRAINING/'input_manifest.json')
FIG_CLASSES = ['anthracnose', 'bacterial_black_spot', 'bacterial_canker', 'healthy_fruit',
               'healthy_leaf', 'mango_scab', 'sooty_mold']
if (FIG_COMPLETE['modelVersion'] != 'model-v4.3.0-candidate' or
    FIG_COMPLETE['classes'] != FIG_CLASSES or FIG_PLAN['classes'] != FIG_CLASSES or
    FIG_MANIFEST['classes'] != FIG_CLASSES or FIG_COMPLETE['armOrder'] != ['combined'] or
    set(FIG_COMPLETE['arms']) != {'combined'} or FIG_PLAN['arms'] != ['combined'] or
    FIG_COMPLETE.get('testPredictionsRun') is not False):
    raise ValueError('Select the completed single seven-class V4.3.0 combined run.')
verify_figure_artifacts(FIG_TRAINING, FIG_COMPLETE['artifacts'])
for name, key in (('plan.json', 'planSha256'), ('input_manifest.json', 'inputManifestSha256'),
                  ('dataset_split.zip', 'datasetSplitSha256'), ('scene_review.json', 'sceneReviewSha256')):
    if sha256_file(FIG_TRAINING/name) != FIG_COMPLETE[key]:
        raise ValueError('Completed figure source contract changed: ' + name)
if ({s: sum(r['subset'] == s for r in FIG_MANIFEST['images']) for s in ('train', 'val', 'test')} !=
    {'train': 1819, 'val': 391, 'test': 393}):
    raise ValueError('Figure input no longer contains the full historical seven-class split.')
with zipfile.ZipFile(FIG_TRAINING/'dataset_split.zip') as bundle:
    names = bundle.namelist()
    if len(names) != len(set(names)) or set(names) != {r['relativePath'] for r in FIG_MANIFEST['images']}:
        raise ValueError('Saved figure ZIP entries differ from the manifest.')
FIGURES_DIR = FIG_TRAINING/'figures'
print('Verified figure source:', FIG_TRAINING)
print('Seven classes:', FIG_CLASSES)
print('Scene sources:', FIG_PLAN['backgroundRoots'], '; figures use verified frozen backgrounds/ copies.')

### Step 26: Draw representative augmentation and seven validation Grad-CAM examples

- **Purpose:** Reconstruct a scheduled train scene/light combination and show trained-model attention on one validation image per class.
- **Reads:** Frozen scene schedule, approved mask, scene bytes, ZIP image identities, and calibrated combined model.
- **Saves:** figures/augmentation_preview.png, figures/grad_cam.png, and training_figure_provenance.json; unique attempt/pending records permit verified publication recovery.
- **Check:** The four columns are views of one photo, not four trained models. Only the Combined input top column reconstructs the scheduled pre-Keras input; other columns isolate effects. Bottom transforms are new draws. Grad-CAM reads validation only and does not establish accuracy.

In [ ]:
# These diagnostics reuse saved train/validation images and weights; no model.fit.
diagnostic_names = ('augmentation_preview.png', 'grad_cam.png')
diagnostic_record_name = 'training_figure_provenance.json'
if (FIG_TRAINING/diagnostic_record_name).exists():
    pre_provenance = read_json(FIG_TRAINING/diagnostic_record_name)
    if pre_provenance['trainingCompleteSha256'] != sha256_file(FIG_TRAINING/'training_complete.json'):
        raise ValueError('Existing training diagnostics belong to a different completion record.')
    verify_figure_artifacts(FIGURES_DIR, pre_provenance['figures'])
    print('Verified existing training diagnostics:', FIGURES_DIR)
elif (FIG_TRAINING/(diagnostic_record_name+'.pending.json')).exists():
    pre_provenance = finish_pending_figure_phase(diagnostic_record_name)
    print('Resumed verified training-diagnostic publication:', FIGURES_DIR)
else:
    if any((FIGURES_DIR/name).exists() for name in diagnostic_names):
        raise ValueError('Untracked partial diagnostic output exists; inspect it before a new phase.')
    diagnostic_attempt = new_figure_attempt('training')
    with zipfile.ZipFile(FIG_TRAINING/'dataset_split.zip') as fig_bundle:
        FIG_PREVIEW = save_augmentation_preview(
            FIG_TRAINING, diagnostic_attempt, FIG_PLAN, FIG_MANIFEST, fig_bundle)
        FIG_GRADCAM = save_grad_cam(
            FIG_TRAINING, diagnostic_attempt, FIG_CLASSES, FIG_MANIFEST, fig_bundle,
            FIG_COMPLETE['arms']['combined']['modelSha256'])
    pre_provenance = {
        'workflowVersion': 'v4.3.0', 'modelVersion': FIG_COMPLETE['modelVersion'],
        'trainingCompleteSha256': sha256_file(FIG_TRAINING/'training_complete.json'),
        'modelSha256': FIG_COMPLETE['arms']['combined']['modelSha256'],
        'planSha256': FIG_COMPLETE['planSha256'], 'inputManifestSha256': FIG_COMPLETE['inputManifestSha256'],
        'sceneReviewSha256': FIG_COMPLETE['sceneReviewSha256'],
        'backgroundScheduleSha256': FIG_COMPLETE['backgroundScheduleSha256'],
        'augmentationPreview': FIG_PREVIEW, 'gradCamValidationImages': FIG_GRADCAM,
        'figures': {'training_curves.png': sha256_file(FIGURES_DIR/'training_curves.png'),
                    **{name: sha256_file(diagnostic_attempt/name) for name in diagnostic_names}},
        'limits': ['Final Keras augmentation transforms are new representative random draws.',
                   'Grad-CAM uses validation images and does not measure accuracy.']}
    publish_figure_phase(diagnostic_attempt, diagnostic_record_name, pre_provenance, diagnostic_names)
    print('Saved training diagnostics:', FIGURES_DIR)

### Step 27: Restore the matching frozen threshold evaluation

- **Purpose:** Require a completed threshold run whose selection/model/evaluation hashes match this exact training run.
- **Reads:** selected_thresholds.json, evaluation.json, threshold_complete.json, saved metric/search CSVs, and training diagnostics provenance.
- **Saves:** Verified report variables only.
- **Check:** Run the threshold notebook first. Its completion artifact ledger must match; an infeasible gate remains infeasible and must never be presented as selected/adoptable.

In [ ]:
# First run the separate V4.3.0 threshold notebook; paste its exact completed path.
FIGURE_THRESHOLD_RUN = ''
if not FIGURE_THRESHOLD_RUN.strip():
    raise ValueError('Finish threshold fitting/evaluation, then set FIGURE_THRESHOLD_RUN.')
threshold_root = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.3.0/threshold_fitting').resolve()
supplied_threshold = Path(FIGURE_THRESHOLD_RUN.strip())
if supplied_threshold.is_absolute():
    FIG_THRESHOLD = supplied_threshold.resolve()
elif supplied_threshold.name == str(supplied_threshold):
    FIG_THRESHOLD = (threshold_root/supplied_threshold).resolve()
else:
    raise ValueError('Use the full Drive path or exact threshold run folder name.')
if FIG_THRESHOLD == threshold_root or not FIG_THRESHOLD.is_relative_to(threshold_root):
    raise ValueError('Use one exact V4.3.0 threshold run, not a parent folder or different experiment.')
selected = read_json(FIG_THRESHOLD/'selected_thresholds.json')
evaluation = read_json(FIG_THRESHOLD/'evaluation.json')
threshold_complete = read_json(FIG_THRESHOLD/'threshold_complete.json')
verify_figure_artifacts(FIG_THRESHOLD, threshold_complete['artifacts'])
if (selected['classes'] != FIG_CLASSES or set(selected['arms']) != {'combined'} or
    selected['trainingCompleteSha256'] != sha256_file(FIG_TRAINING/'training_complete.json') or
    evaluation['selectionSha256'] != sha256_file(FIG_THRESHOLD/'selected_thresholds.json') or
    selected['arms']['combined']['modelSha256'] != FIG_COMPLETE['arms']['combined']['modelSha256'] or
    evaluation['arms']['combined']['modelSha256'] != FIG_COMPLETE['arms']['combined']['modelSha256']):
    raise ValueError('Frozen threshold outputs do not belong to this exact completed training model.')
if (threshold_complete['trainingCompleteSha256'] != selected['trainingCompleteSha256'] or
    threshold_complete['selectedThresholdsSha256'] != sha256_file(FIG_THRESHOLD/'selected_thresholds.json') or
    threshold_complete['evaluationSha256'] != sha256_file(FIG_THRESHOLD/'evaluation.json')):
    raise ValueError('Threshold completion provenance differs from selected/evaluated evidence.')
if not (FIG_TRAINING/'training_figure_provenance.json').is_file():
    raise FileNotFoundError('Finish Step 26 training diagnostics before final figure provenance.')
training_figure_provenance = read_json(FIG_TRAINING/'training_figure_provenance.json')
if training_figure_provenance['trainingCompleteSha256'] != sha256_file(FIG_TRAINING/'training_complete.json'):
    raise ValueError('Training diagnostic provenance belongs to another run.')
verify_figure_artifacts(FIGURES_DIR, training_figure_provenance['figures'])
print('Verified frozen threshold selection and evaluation:', FIG_THRESHOLD)
print('Selected gate status:', selected['arms']['combined']['status'])
print('Final plots will only read saved numbers; no test/OOD prediction or threshold refit occurs here.')

### Step 28: Draw confusion, per-class metrics, and threshold-search charts

- **Purpose:** Turn complete frozen seven-class test numbers and saved tune-only search results into figures.
- **Reads:** Raw evaluation confusion/report, raw_per_class_metrics.csv, and threshold_search.csv from the verified threshold run.
- **Saves:** figures/confusion_matrix.png, per_class_metrics.png, per_class_metrics.csv, threshold_search.png, and figure_provenance.json.
- **Check:** Raw confusion must total 393 images with seven axes; CSV scores/support must match raw evaluation JSON. Rejection plots/search are separate from raw class accuracy. No inference or fitting occurs.

In [ ]:
# Keep a failed rendering attempt isolated; retry or finish a verified pending publication.
FINAL_FIGURES_DIR = FIG_TRAINING/'figures'
FIGURES_DIR = FINAL_FIGURES_DIR
if (FIG_TRAINING/'figure_provenance.json').exists():
    saved = read_json(FIG_TRAINING/'figure_provenance.json')
    if (saved['trainingCompleteSha256'] != sha256_file(FIG_TRAINING/'training_complete.json') or
        saved['selectedThresholdsSha256'] != sha256_file(FIG_THRESHOLD/'selected_thresholds.json') or
        saved['evaluationSha256'] != sha256_file(FIG_THRESHOLD/'evaluation.json')):
        raise ValueError('Existing evaluation figures belong to different frozen reports.')
    verify_figure_artifacts(FINAL_FIGURES_DIR, saved['figures'])
    if sha256_file(FINAL_FIGURES_DIR/'per_class_metrics.csv') != saved['perClassCsvSha256']:
        raise ValueError('Saved per-class figure CSV changed.')
    print('Verified existing evaluation figures:', FINAL_FIGURES_DIR)
elif (FIG_TRAINING/'figure_provenance.json.pending.json').exists():
    finish_pending_figure_phase('figure_provenance.json')
    print('Resumed verified evaluation-figure publication:', FINAL_FIGURES_DIR)
else:
    final_names = ('confusion_matrix.png', 'per_class_metrics.png', 'per_class_metrics.csv', 'threshold_search.png')
    if any((FINAL_FIGURES_DIR/name).exists() for name in final_names):
        raise ValueError('Untracked partial final chart exists; inspect it before a new attempt.')
    evaluation_attempt = new_figure_attempt('evaluation')
    FIGURES_DIR = evaluation_attempt
    try:
        chart_names = ("confusion_matrix.png", "per_class_metrics.png", "per_class_metrics.csv", "threshold_search.png")

        classes = FIG_COMPLETE["classes"]
        raw_matrix = np.asarray(evaluation["arms"]["combined"]["rawConfusionMatrix"], dtype=float)
        if not np.isfinite(raw_matrix).all() or np.any(raw_matrix < 0) or not np.array_equal(raw_matrix, raw_matrix.astype(int)):
            raise ValueError("Raw confusion counts must be finite nonnegative integers.")
        matrix = raw_matrix.astype(int)
        test_count = sum(1 for row in FIG_MANIFEST["images"] if row["subset"] == "test")
        if matrix.shape != (len(classes), len(classes)) or matrix.sum() != test_count or test_count != 393:
            raise ValueError("Raw confusion matrix does not match the 393 frozen seven-class test photos.")
        fig, ax = plt.subplots(figsize=(11, 10), constrained_layout=True)
        ax.imshow(matrix, cmap="Blues", vmin=0, vmax=max(1, int(matrix.max())))
        short = [name.replace("_", " ") for name in classes]
        ax.set(xticks=range(len(classes)), yticks=range(len(classes)),
               xticklabels=short, yticklabels=short, xlabel="Predicted", ylabel="Actual",
               title="Seven-class combined raw test confusion matrix (before rejection)")
        plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
        for row in range(len(classes)):
            for col in range(len(classes)):
                value = int(matrix[row, col])
                ax.text(col, row, str(value), ha="center", va="center",
                        color="white" if value > matrix.max() / 2 else "black")
        fig.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=160)
        plt.close(fig)

        with (FIG_THRESHOLD / "arms/combined/raw_per_class_metrics.csv").open(newline="", encoding="utf-8") as handle:
            metrics = list(csv.DictReader(handle))
        report = evaluation["arms"]["combined"]["rawClassificationReport"]
        for row in metrics:
            label = row["class"]
            for metric in ("precision", "recall", "f1-score", "support"):
                value = float(row[metric])
                if not np.isfinite(value) or not np.isclose(value, float(report[label][metric]), atol=1e-8):
                    raise ValueError("Per-class CSV differs from frozen raw evaluation JSON.")
            if int(float(row["support"])) != matrix[classes.index(label)].sum():
                raise ValueError("Raw per-class support disagrees with confusion matrix.")
        if [row["class"] for row in metrics] != classes:
            raise ValueError("Raw per-class metric order disagrees with model class order.")
        fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
        y = np.arange(len(classes))
        for offset, metric in enumerate(("precision", "recall", "f1-score")):
            ax.barh(y + (offset - 1) * 0.23,
                    [float(row[metric]) for row in metrics], height=0.22, label=metric)
        ax.set(yticks=y, yticklabels=short, xlim=(0, 1.02),
               xlabel="Raw test metric (before rejection)", title="Seven-class combined per-class results")
        ax.invert_yaxis()
        ax.legend()
        ax.grid(axis="x", alpha=0.2)
        fig.savefig(FIGURES_DIR / "per_class_metrics.png", dpi=160)
        plt.close(fig)
        with (FIGURES_DIR / "per_class_metrics.csv").open("x", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=["class", "precision", "recall", "f1-score", "support"])
            writer.writeheader()
            writer.writerows(metrics)

        gate = selected["arms"]["combined"]
        margin = gate["thresholds"]["marginThreshold"] if gate["status"] == "feasible" else 0.0
        with (FIG_THRESHOLD / "arms/combined/threshold_search.csv").open(newline="", encoding="utf-8") as handle:
            search = [row for row in csv.DictReader(handle) if float(row["margin"]) == margin]
        if not search:
            raise ValueError("Threshold search has no rows for displayed margin.")
        search.sort(key=lambda row: float(row["entropy"]))
        fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
        x = [float(row["entropy"]) for row in search]
        ax.plot(x, [int(row["valAccepted"]) / int(row["valTotal"]) for row in search],
                label="validation mango retained")
        ax.plot(x, [int(row["oodRejected"]) / int(row["oodTotal"]) for row in search],
                label="old OOD tune rejected")
        ax.axhline(0.95, color="gray", linestyle="--", label="95% retention target")
        if gate["status"] == "feasible":
            ax.axvline(gate["thresholds"]["entropyThreshold"], color="crimson",
                       linestyle=":", label="selected entropy")
        else:
            ax.text(0.02, 0.03, "No feasible selected gate; showing margin 0", transform=ax.transAxes)
        ax.set(xlabel=f"Maximum entropy at margin = {margin:.2f}", ylabel="Fraction",
               ylim=(-0.02, 1.02), title="Seven-class combined threshold search (old OOD tune only)")
        ax.grid(alpha=0.2)
        ax.legend()
        fig.savefig(FIGURES_DIR / "threshold_search.png", dpi=160)
        plt.close(fig)

        figure_provenance = {
            "status": "experimental candidate figures; not adoption evidence",
            "workflowVersion": "v4.3.0", "modelVersion": FIG_COMPLETE["modelVersion"],
            "trainingRun": str(FIG_TRAINING), "thresholdRun": str(FIG_THRESHOLD),
            "modelSha256": FIG_COMPLETE["arms"]["combined"]["modelSha256"],
            "trainingCompleteSha256": sha256_file(FIG_TRAINING / "training_complete.json"),
            "selectedThresholdsSha256": sha256_file(FIG_THRESHOLD / "selected_thresholds.json"),
            "thresholdCompleteSha256": sha256_file(FIG_THRESHOLD / "threshold_complete.json"),
            "evaluationSha256": sha256_file(FIG_THRESHOLD / "evaluation.json"),
            "trainingFigureProvenanceSha256": sha256_file(FIG_TRAINING / "training_figure_provenance.json"),
            "sourceCsvSha256": {
                "perClassMetrics": sha256_file(FIG_THRESHOLD / "arms/combined/raw_per_class_metrics.csv"),
                "thresholdSearch": sha256_file(FIG_THRESHOLD / "arms/combined/threshold_search.csv")},
            "figures": {name: sha256_file((FIGURES_DIR if name in chart_names else FINAL_FIGURES_DIR)/name) for name in ("training_curves.png", "augmentation_preview.png", "grad_cam.png", "confusion_matrix.png", "per_class_metrics.png", "threshold_search.png")},
            "perClassCsvSha256": sha256_file(FIGURES_DIR / "per_class_metrics.csv"),
            "limits": ["Historical test/OOD data are reused, not fresh field validation.",
                       "Threshold charts display saved search results and do not refit gates."]}
        publish_figure_phase(evaluation_attempt, "figure_provenance.json", figure_provenance, chart_names)
        print("Saved six seven-class combined figures and per_class_metrics.csv:", FIGURES_DIR)
    finally:
        FIGURES_DIR = FINAL_FIGURES_DIR

### Step 29: Inspect the saved figures and their identities

- **Purpose:** Review the six integrated figures and verify their evidence hashes before curating a report.
- **Reads:** figure_provenance.json and all saved figure files.
- **Saves:** Displayed figures only.
- **Check:** Inspect all seven class labels, augmentation captions, validation-only Grad-CAM, curve history, complete raw supports, and any infeasible gate warning.

In [ ]:
from IPython.display import display
figure_names = ('training_curves.png', 'augmentation_preview.png', 'grad_cam.png',
                'confusion_matrix.png', 'per_class_metrics.png', 'threshold_search.png')
provenance = read_json(FIG_TRAINING/'figure_provenance.json')
verify_figure_artifacts(FIGURES_DIR, provenance['figures'])
for name in figure_names:
    print(name, provenance['figures'][name])
    with Image.open(FIGURES_DIR/name) as picture:
        display(picture.copy())
print('Full raw class metrics CSV:', FIGURES_DIR/'per_class_metrics.csv')
print('Figure provenance:', FIG_TRAINING/'figure_provenance.json')
print('Historical test/OOD evidence is reused, not independent field validation.')

### Step 30: Keep the complete candidate evidence together

- **Purpose:** Hand off a reviewable experimental candidate without implying accuracy approval or deployment.
- **Reads:** Both exact Drive run IDs, their completion hashes, frozen model/config hashes, all review decisions, figures, and reports.
- **Saves:** No automatic repository or application writes. Curate evidence only after actually running and checking both notebooks.
- **Check:** Record candidate commit ID, full source/training/threshold paths, runtime versions, scene 62/138 expectations versus actual readable/unique/approved counts, white/mask counts and reuse evidence, model SHA-256, selected threshold feasibility, raw and gated metrics, and known reused-data limitations. Preserve completed historical notebooks and evidence.

Keep the entire training run and threshold run on Drive: curated PNG/JSON/CSV files do not replace source images, reviewed masks, the calibrated model, candidate browser files, or per-image predictions. `training_complete.json` freezes training artifacts; `training_figure_provenance.json` and `figure_provenance.json` describe diagnostics added afterward without changing that completion record. `threshold_complete.json` freezes threshold/report evidence. A fresh-runtime figure session needs the saved run paths only; it reads frozen scene copies from both original pools.

The threshold run's `candidate_models/combined/` is a potential future browser candidate after evaluation. Nothing here installs it into `public/model/`. App reliability/physical-device checks, independent accuracy/OOD evaluation, explicit release approval, and promotion of the exact tested deployment remain separate tasks. This notebook template has no completed V4.3.0 numerical results.